In [1]:
# %%
# =========================
# Cell 0 (UPDATED)
# Determinism, /data-first outputs,
# SAFE sampler loader (avoid rwkv_medsam2/__init__.py -> ext.vrwkv),
# + SAFE vis.py loader (train_sam2.py-style)
# + SAFE metric-helper loader + local compatibility wrappers for
#   compute_metrics_2d / compute_metrics_3d built from current func_metrics.py
# =========================
import os, json, random, re, time, math
from glob import glob
from collections import Counter
from datetime import datetime

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

import SimpleITK as sitk
from PIL import Image

SEED = 42

def set_determinism(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

set_determinism(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

PREPROC_ROOT = "/data/Preprocessed"
assert os.path.isdir(PREPROC_ROOT), f"Missing: {PREPROC_ROOT}"

TARGET_HW = 512  # keep identical

# /data-first outputs
RUN_NAME = datetime.now().strftime("oxford_refine_%Y%m%d_%H%M%S")
OUT_ROOT = f"/data/jrbonill/oxford_refine_runs/{RUN_NAME}"
TB_DIR   = f"{OUT_ROOT}/tb"
CKPT_OUT = f"{OUT_ROOT}/checkpoints"
TMP_DIR  = f"{OUT_ROOT}/tmp"
VIS_DIR  = f"{OUT_ROOT}/vis"

os.makedirs(TB_DIR, exist_ok=True)
os.makedirs(CKPT_OUT, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)
os.makedirs(VIS_DIR, exist_ok=True)

os.environ["TMPDIR"] = TMP_DIR
os.environ["XDG_CACHE_HOME"] = f"{OUT_ROOT}/cache"
os.makedirs(os.environ["XDG_CACHE_HOME"], exist_ok=True)

print("OUT_ROOT:", OUT_ROOT)
print("TB_DIR  :", TB_DIR)
print("CKPT_OUT:", CKPT_OUT)
print("VIS_DIR :", VIS_DIR)

# ---- SAFE loader for BalancedTaskBatchSampler (bypasses rwkv_medsam2/__init__.py -> ext.vrwkv)
import sys, importlib.util, importlib.machinery

RWKV_ROOT_CANDIDATES = [
    "/data/jrbonill/RWKV-MedSAM2",
    "/data/jrbonill/rwkv_medsam2",
    "/RWKV-MedSAM2",  # your traceback shows this path
]

def _find_rwkv_dataset_py():
    for root in RWKV_ROOT_CANDIDATES:
        ds_path = os.path.join(root, "rwkv_medsam2", "dataset.py")
        if os.path.isfile(ds_path):
            return ds_path
    return None

def load_balanced_task_batch_sampler():
    """
    Load BalancedTaskBatchSampler without importing rwkv_medsam2/__init__.py,
    so we don't require ext.vrwkv in this notebook environment.
    """
    try:
        from rwkv_medsam2.dataset import BalancedTaskBatchSampler
        return BalancedTaskBatchSampler
    except Exception as e:
        ds_path = _find_rwkv_dataset_py()
        if ds_path is None:
            raise RuntimeError(
                "Could not find rwkv_medsam2/dataset.py in RWKV_ROOT_CANDIDATES. "
                "Update RWKV_ROOT_CANDIDATES to your RWKV-MedSAM2 repo path."
            ) from e

        name = "rwkv_medsam2_dataset_standalone"
        loader = importlib.machinery.SourceFileLoader(name, ds_path)
        spec = importlib.util.spec_from_loader(loader.name, loader)
        mod = importlib.util.module_from_spec(spec)
        loader.exec_module(mod)

        if not hasattr(mod, "BalancedTaskBatchSampler"):
            raise RuntimeError(f"BalancedTaskBatchSampler not found in: {ds_path}")
        return mod.BalancedTaskBatchSampler

BalancedTaskBatchSampler = load_balanced_task_batch_sampler()
print("Loaded BalancedTaskBatchSampler:", BalancedTaskBatchSampler)

# ---- SAFE loader for vis.py (so we do not depend on `import vis` working)
VIS_PY_CANDIDATES = [
    os.path.join(os.getcwd(), "vis.py"),
    "./vis.py",
    "/data/jrbonill/oxford_medsam2_env/Medical-SAM2/vis.py",
    "/data/jrbonill/oxford_medsam2_env/site/vis.py",
    "/data/jrbonill/oxford_medsam2_env/vis.py",
    "/data/jrbonill/vis.py",
    "/RWKV-MedSAM2/rwkv_medsam2/utils/vis.py",
    "/mnt/data/vis.py",
]

def _find_vis_py():
    for p in VIS_PY_CANDIDATES:
        try:
            if p and os.path.isfile(p):
                return os.path.abspath(p)
        except Exception:
            pass
    return None

def load_vis_helpers():
    vis_path = _find_vis_py()
    if vis_path is None:
        raise RuntimeError(
            "Could not locate vis.py. Put vis.py next to this notebook, "
            "or add its absolute path to VIS_PY_CANDIDATES in Cell 0."
        )

    name = "vis_standalone"
    loader = importlib.machinery.SourceFileLoader(name, vis_path)
    spec = importlib.util.spec_from_loader(loader.name, loader)
    mod = importlib.util.module_from_spec(spec)
    loader.exec_module(mod)

    needed = ["make_vis_frames", "save_vis_gif", "frames_to_tb_video"]
    missing = [k for k in needed if not hasattr(mod, k)]
    if missing:
        raise RuntimeError(f"vis.py at {vis_path} is missing: {missing}")

    return mod.make_vis_frames, mod.save_vis_gif, mod.frames_to_tb_video, vis_path

make_vis_frames, save_vis_gif, frames_to_tb_video, VIS_PY_PATH = load_vis_helpers()
print("Loaded vis.py from:", VIS_PY_PATH)
print("vis helpers:", make_vis_frames, save_vis_gif, frames_to_tb_video)

# =========================
# Cell 0C — SAFE loader for TaskAggregator / ModalityAggregator
# + metric helper functions from the current func_metrics.py
# + notebook-local compatibility wrappers compute_metrics_2d / compute_metrics_3d
# =========================
import importlib.util, importlib.machinery, os

def _find_rwkv_func_metrics_py():
    for root in RWKV_ROOT_CANDIDATES:
        p = os.path.join(root, "rwkv_medsam2", "functions", "func_metrics.py")
        if os.path.isfile(p):
            return p
    return None

def load_metrics_utils():
    wanted = [
        "TaskAggregator", "ModalityAggregator",
        "dice_iou", "ece", "try_auc", "fg_bin", "sigmoid_np",
    ]
    try:
        from rwkv_medsam2.functions.func_metrics import (
            TaskAggregator, ModalityAggregator,
            dice_iou, ece, try_auc, fg_bin, sigmoid_np,
        )
        return TaskAggregator, ModalityAggregator, dice_iou, ece, try_auc, fg_bin, sigmoid_np
    except Exception as e:
        fm_path = _find_rwkv_func_metrics_py()
        if fm_path is None:
            raise RuntimeError(
                "Could not find rwkv_medsam2/functions/func_metrics.py in RWKV_ROOT_CANDIDATES. "
                "Update RWKV_ROOT_CANDIDATES to your RWKV-MedSAM2 repo path."
            ) from e

        name = "rwkv_medsam2_func_metrics_standalone"
        loader = importlib.machinery.SourceFileLoader(name, fm_path)
        spec = importlib.util.spec_from_loader(loader.name, loader)
        mod = importlib.util.module_from_spec(spec)
        loader.exec_module(mod)

        missing = [k for k in wanted if not hasattr(mod, k)]
        if missing:
            raise RuntimeError(f"Missing {missing} in: {fm_path}")

        return (
            mod.TaskAggregator,
            mod.ModalityAggregator,
            mod.dice_iou,
            mod.ece,
            mod.try_auc,
            mod.fg_bin,
            mod.sigmoid_np,
        )

(
    TaskAggregator,
    ModalityAggregator,
    dice_iou,
    ece,
    try_auc,
    fg_bin,
    sigmoid_np,
) = load_metrics_utils()

print("Loaded metric utilities:", TaskAggregator, ModalityAggregator, dice_iou, ece, try_auc, fg_bin, sigmoid_np)

_METRIC_THR_PROBS = np.linspace(0.05, 0.95, 19, dtype=np.float64)
_METRIC_THR_LOGITS = np.log(_METRIC_THR_PROBS / (1.0 - _METRIC_THR_PROBS)).astype(np.float64)

def _as_numpy_bool_mask(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    return (x > 0.5).astype(np.uint8)

def _as_numpy_probs(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    return x.astype(np.float32)

def _slice_by_idx(arr, idx_list=None, n_valid=None):
    arr = np.asarray(arr)
    if idx_list is not None:
        idx = np.asarray(list(idx_list), dtype=np.int64)
        if idx.size > 0:
            arr = arr[idx]
    elif n_valid is not None:
        arr = arr[:int(n_valid)]
    return arr

def _metrics_from_logits_and_gt(logits_np, gt_np):
    logits_np = np.asarray(logits_np, dtype=np.float32)
    gt_np = np.asarray(gt_np, dtype=np.uint8)

    if logits_np.ndim == 2:
        logits_np = logits_np[None, ...]
    if gt_np.ndim == 2:
        gt_np = gt_np[None, ...]

    probs_np = sigmoid_np(logits_np)
    pred05 = (logits_np > 0.0).astype(np.uint8)

    dices = []
    ious = []
    fg_fracs = []
    for kslice in range(gt_np.shape[0]):
        d, i = dice_iou(torch.from_numpy(pred05[kslice]), torch.from_numpy(gt_np[kslice]))
        dices.append(float(d))
        ious.append(float(i))
        fg_fracs.append(float(gt_np[kslice].mean()))
    dice05 = float(np.mean(dices)) if dices else float("nan")
    iou05 = float(np.mean(ious)) if ious else float("nan")

    avg_d_list = []
    avg_i_list = []
    best_d = -1.0
    best_i = -1.0
    best_thr = float(_METRIC_THR_LOGITS[0])

    for thr in _METRIC_THR_LOGITS:
        pred = (logits_np > float(thr)).astype(np.uint8)
        d_thr = []
        i_thr = []
        for kslice in range(gt_np.shape[0]):
            d, i = dice_iou(torch.from_numpy(pred[kslice]), torch.from_numpy(gt_np[kslice]))
            d_thr.append(float(d))
            i_thr.append(float(i))
        d_m = float(np.mean(d_thr)) if d_thr else float("nan")
        i_m = float(np.mean(i_thr)) if i_thr else float("nan")
        avg_d_list.append(d_m)
        avg_i_list.append(i_m)
        if d_m > best_d:
            best_d = d_m
            best_i = i_m
            best_thr = float(thr)

    dice_avg = float(np.mean(avg_d_list)) if avg_d_list else float("nan")
    iou_avg = float(np.mean(avg_i_list)) if avg_i_list else float("nan")

    flat_probs = probs_np.reshape(-1).astype(np.float64)
    flat_gt = gt_np.reshape(-1).astype(np.uint8)
    ece_val = float(ece(flat_probs, flat_gt, n_bins=15)) if flat_probs.size > 0 else float("nan")
    roc_auc, pr_auc = try_auc(flat_gt, flat_probs) if flat_probs.size > 0 else (float("nan"), float("nan"))
    roc_auc = float(roc_auc) if roc_auc == roc_auc else float("nan")
    pr_auc = float(pr_auc) if pr_auc == pr_auc else float("nan")

    out = {
        "dice@0.5": dice05,
        "iou@0.5": iou05,
        "dice@avg_thr": dice_avg,
        "iou@avg_thr": iou_avg,
        "dice_best": float(best_d),
        "iou_best": float(best_i),
        "best_thr": float(best_thr),
        "ece": ece_val,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "n": int(gt_np.shape[0]),
    }

    for frac, d, i in zip(fg_fracs, dices, ious):
        key = fg_bin(frac)
        out[f"{key}/dice@0.5"] = float(d)
        out[f"{key}/iou@0.5"] = float(i)

    return out, probs_np, gt_np

def compute_metrics_2d(probs_flat, masks_flat, idx_list=None, n_valid=None, **kwargs):
    probs_np = _slice_by_idx(_as_numpy_probs(probs_flat), idx_list=idx_list, n_valid=n_valid)
    gt_np = _slice_by_idx(_as_numpy_bool_mask(masks_flat), idx_list=idx_list, n_valid=n_valid)

    if probs_np.ndim == 2:
        probs_np = probs_np[None, ...]
    if gt_np.ndim == 2:
        gt_np = gt_np[None, ...]

    probs_np = np.clip(probs_np.astype(np.float32), 1e-6, 1.0 - 1e-6)
    logits_np = np.log(probs_np / (1.0 - probs_np)).astype(np.float32)

    out, _, gt_np = _metrics_from_logits_and_gt(logits_np, gt_np)
    out["n"] = int(gt_np.shape[0])
    return out

def compute_metrics_3d(probs, gts, idx_list=None, n_valid=None, batch=None, **kwargs):
    probs_np = _slice_by_idx(_as_numpy_probs(probs), idx_list=idx_list, n_valid=n_valid)
    gt_np = _slice_by_idx(_as_numpy_bool_mask(gts), idx_list=idx_list, n_valid=n_valid)

    if probs_np.ndim == 2:
        probs_np = probs_np[None, ...]
    if gt_np.ndim == 2:
        gt_np = gt_np[None, ...]

    probs_np = np.clip(probs_np.astype(np.float32), 1e-6, 1.0 - 1e-6)
    logits_np = np.log(probs_np / (1.0 - probs_np)).astype(np.float32)

    out, _, gt_np = _metrics_from_logits_and_gt(logits_np, gt_np)

    out["dice3d@0.5"] = float(out["dice@0.5"])
    out["iou3d@0.5"] = float(out["iou@0.5"])
    out["dice3d@best"] = float(out["dice_best"])
    out["iou3d@best"] = float(out["iou_best"])

    vol_mask = (gt_np.sum(axis=(1, 2)) > 0)
    if np.any(vol_mask):
        vL = logits_np[vol_mask]
        vG = gt_np[vol_mask]
        best_dv = -1.0
        best_iv = -1.0
        best_tv = float(_METRIC_THR_LOGITS[0])
        for thr in _METRIC_THR_LOGITS:
            predv = (vL > float(thr)).astype(np.uint8)
            d, i = dice_iou(torch.from_numpy(predv), torch.from_numpy(vG))
            d = float(d)
            i = float(i)
            if d > best_dv:
                best_dv = d
                best_iv = i
                best_tv = float(thr)
        out["dice3d@best_vol"] = float(best_dv)
        out["iou3d@best_vol"] = float(best_iv)
        out["best_thr3d"] = float(best_tv)
    else:
        out["dice3d@best_vol"] = float("nan")
        out["iou3d@best_vol"] = float("nan")
        out["best_thr3d"] = float("nan")

    out["n"] = int(gt_np.shape[0])
    out["n_vol"] = 1
    return out

print("Defined compatibility wrappers:", compute_metrics_2d, compute_metrics_3d)

import warnings
import re

warnings.filterwarnings(
    "ignore",
    message=re.escape("Deterministic behavior was enabled") + ".*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=re.escape("Memory Efficient attention defaults to a non-deterministic algorithm") + ".*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=re.escape("Flash Attention defaults to a non-deterministic algorithm") + ".*",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message=re.escape("upsample_bicubic2d_backward_out_cuda does not have a deterministic implementation") + ".*",
    category=UserWarning,
)


DEVICE: cuda
OUT_ROOT: /data/jrbonill/oxford_refine_runs/oxford_refine_20260312_023627
TB_DIR  : /data/jrbonill/oxford_refine_runs/oxford_refine_20260312_023627/tb
CKPT_OUT: /data/jrbonill/oxford_refine_runs/oxford_refine_20260312_023627/checkpoints
VIS_DIR : /data/jrbonill/oxford_refine_runs/oxford_refine_20260312_023627/vis


/usr/local/lib/python3.10/dist-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.10/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float64'> type is zero.
  return self._float_to_str(self.smallest_subnormal)
/usr/local/lib/python3.10/dist-packages/numpy/core/getlimits.py:518: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  setattr(self, word, getattr(machar, word).flat[0])
/usr/local/lib/python3.10/dist-packages/numpy/core/getlimits.py:89: UserWarning: The value of the smallest subnormal for <class 'numpy.float32'> type is zero.
  return self._float_to_str(self.smallest_subnormal)


Loaded BalancedTaskBatchSampler: <class 'rwkv_medsam2_dataset_standalone.BalancedTaskBatchSampler'>
Loaded vis.py from: /RWKV-MedSAM2/rwkv_medsam2/utils/vis.py
vis helpers: <function make_vis_frames at 0x7f7a69f3c430> <function save_vis_gif at 0x7f7a69f3c4c0> <function frames_to_tb_video at 0x7f7a69f3c550>
Loaded metric utilities: <class 'rwkv_medsam2_func_metrics_standalone.TaskAggregator'> <class 'rwkv_medsam2_func_metrics_standalone.ModalityAggregator'> <function dice_iou at 0x7f7a691fea70> <function ece at 0x7f7a691fed40> <function try_auc at 0x7f7a691fedd0> <function fg_bin at 0x7f7a691ff760> <function sigmoid_np at 0x7f7a691ff640>
Defined compatibility wrappers: <function compute_metrics_2d at 0x7f7a691ffac0> <function compute_metrics_3d at 0x7f7a691ffb50>


In [2]:
# %%
# =========================
# Cell 1 (REPLACEMENT)
# Load the EXACT same saved loaders used by training.ipynb
# WITHOUT importing rwkv_medsam2/__init__.py or ext/*
# Robust to different pickle key names
# =========================
import os
import sys
import pickle
import types
import importlib.util
import importlib.machinery
from collections import Counter

LOADERS_PKL = "/data/loaders32.pkl"
assert os.path.exists(LOADERS_PKL), f"Missing saved loaders: {LOADERS_PKL}"

def _find_repo_root():
    for root in RWKV_ROOT_CANDIDATES:
        if os.path.isdir(root) and os.path.isfile(os.path.join(root, "rwkv_medsam2", "dataset.py")):
            return os.path.abspath(root)
    return None

REPO_ROOT = _find_repo_root()
assert REPO_ROOT is not None, (
    "Could not locate RWKV-MedSAM2 repo root from RWKV_ROOT_CANDIDATES."
)

PKG_ROOT = os.path.join(REPO_ROOT, "rwkv_medsam2")
assert os.path.isdir(PKG_ROOT), f"Missing package dir: {PKG_ROOT}"

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

print("[REPO_ROOT]", REPO_ROOT)
print("[PKG_ROOT ]", PKG_ROOT)

def _load_module_from_path(module_name, path):
    loader = importlib.machinery.SourceFileLoader(module_name, path)
    spec = importlib.util.spec_from_loader(loader.name, loader)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    loader.exec_module(mod)
    return mod

def _install_safe_rwkv_package():
    """
    Create a fake rwkv_medsam2 package in sys.modules so pickle can import
    rwkv_medsam2.dataset etc. without executing rwkv_medsam2/__init__.py.
    """
    for name in list(sys.modules.keys()):
        if name == "rwkv_medsam2" or name.startswith("rwkv_medsam2."):
            del sys.modules[name]

    pkg = types.ModuleType("rwkv_medsam2")
    pkg.__path__ = [PKG_ROOT]
    pkg.__file__ = os.path.join(PKG_ROOT, "__init__.py")
    sys.modules["rwkv_medsam2"] = pkg

    dataset_py = os.path.join(PKG_ROOT, "dataset.py")
    assert os.path.isfile(dataset_py), f"Missing: {dataset_py}"
    dataset_mod = _load_module_from_path("rwkv_medsam2.dataset", dataset_py)
    setattr(pkg, "dataset", dataset_mod)

_install_safe_rwkv_package()

print("[SAFE PACKAGE INSTALLED]")
print("  rwkv_medsam2 in sys.modules:", "rwkv_medsam2" in sys.modules)
print("  rwkv_medsam2.dataset loaded :", "rwkv_medsam2.dataset" in sys.modules)

def _looks_like_dataloader(x):
    """
    Heuristic: saved torch DataLoader-like object.
    """
    return (
        hasattr(x, "dataset")
        and hasattr(x, "batch_sampler")
        and hasattr(x, "__iter__")
        and hasattr(x, "__len__")
    )

def _extract_loaders_from_dict(obj):
    """
    Robust extraction from arbitrary dict structure.
    Priority:
      1) common explicit keys
      2) any top-level values that look like loaders
      3) nested dict values that look like loaders
    """
    preferred_key_sets = [
        ("train_loader", "val_loader", "test_loader"),
        ("train", "val", "test"),
        ("train_dataloader", "val_dataloader", "test_dataloader"),
    ]

    for ks in preferred_key_sets:
        if all(k in obj for k in ks):
            return obj[ks[0]], obj[ks[1]], obj[ks[2]]

    # Top-level loader-like values
    loader_like = {k: v for k, v in obj.items() if _looks_like_dataloader(v)}
    if len(loader_like) >= 3:
        print("[PICKLE] top-level loader-like keys:", list(loader_like.keys()))

        def _pick_by_name(name_candidates):
            for k, v in loader_like.items():
                kl = str(k).lower()
                if any(tok in kl for tok in name_candidates):
                    return v
            return None

        train_loader = _pick_by_name(["train"])
        val_loader   = _pick_by_name(["val", "valid"])
        test_loader  = _pick_by_name(["test"])

        if train_loader is not None and val_loader is not None and test_loader is not None:
            return train_loader, val_loader, test_loader

        vals = list(loader_like.values())
        return vals[0], vals[1], vals[2]

    # Nested dicts
    for k, v in obj.items():
        if isinstance(v, dict):
            nested_loader_like = {kk: vv for kk, vv in v.items() if _looks_like_dataloader(vv)}
            if len(nested_loader_like) >= 3:
                print(f"[PICKLE] nested loader-like keys under '{k}':", list(nested_loader_like.keys()))

                def _pick_nested(name_candidates):
                    for kk, vv in nested_loader_like.items():
                        kkl = str(kk).lower()
                        if any(tok in kkl for tok in name_candidates):
                            return vv
                    return None

                train_loader = _pick_nested(["train"])
                val_loader   = _pick_nested(["val", "valid"])
                test_loader  = _pick_nested(["test"])

                if train_loader is not None and val_loader is not None and test_loader is not None:
                    return train_loader, val_loader, test_loader

                vals = list(nested_loader_like.values())
                return vals[0], vals[1], vals[2]

    return None, None, None

def _direct_load_data_loaders(pkl_path):
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)

    print("[PICKLE] top-level type:", type(obj))

    if isinstance(obj, (tuple, list)) and len(obj) == 3:
        return obj[0], obj[1], obj[2]

    if isinstance(obj, dict):
        print("[PICKLE] top-level keys:", list(obj.keys()))
        train_loader, val_loader, test_loader = _extract_loaders_from_dict(obj)
        if train_loader is not None and val_loader is not None and test_loader is not None:
            return train_loader, val_loader, test_loader

    raise RuntimeError(
        f"Could not locate train/val/test loaders inside {pkl_path}. "
        "See the printed top-level keys above."
    )

train_loader_base, val_loader_base, test_loader_base = _direct_load_data_loaders(LOADERS_PKL)

print("\n[BASE LOADERS]")
print("  Train batches:", len(train_loader_base))
print("  Val batches  :", len(val_loader_base))
print("  Test batches :", len(test_loader_base))

# -------------------------
# Apply the same exclusion logic as training.ipynb
# -------------------------
EXCLUDED_DATASET = "wanglab__CT_DeepLesion-MedSAM2"

def _norm_str(x):
    return str(x).strip() if x is not None else ""

def exclude_sequences_by_dataset(seqs, excluded_dataset, case_insensitive=False):
    ed = _norm_str(excluded_dataset)
    if case_insensitive:
        ed = ed.lower()

    kept = []
    removed = 0
    for s in seqs:
        ds = _norm_str(s.get("dataset"))
        if case_insensitive:
            ds = ds.lower()
        if ds == ed:
            removed += 1
            continue
        kept.append(s)
    return kept, removed

def apply_exclusion_to_loader(loader, excluded_dataset):
    before = len(loader.dataset.sequences)
    new_seqs, removed = exclude_sequences_by_dataset(
        loader.dataset.sequences,
        excluded_dataset,
        case_insensitive=False,
    )
    loader.dataset.sequences = new_seqs

    if hasattr(loader.dataset, "entry_dims"):
        loader.dataset.entry_dims = [s["dim"] for s in new_seqs]
    if hasattr(loader.dataset, "entry_modalities"):
        try:
            loader.dataset.entry_modalities = [s.get("modality", "unknown") for s in new_seqs]
        except Exception:
            pass

    after = len(loader.dataset.sequences)
    return before, after, removed

print("\n[EXCLUDE] Before counts:")
for name, loader in [("TRAIN", train_loader_base), ("VAL", val_loader_base), ("TEST", test_loader_base)]:
    c = Counter(_norm_str(s.get("dataset")) for s in loader.dataset.sequences)
    print(name, "unique datasets =", len(c), "| excluded_count =", c.get(EXCLUDED_DATASET, 0))

bt, at, rt = apply_exclusion_to_loader(train_loader_base, EXCLUDED_DATASET)
bv, av, rv = apply_exclusion_to_loader(val_loader_base, EXCLUDED_DATASET)
bq, aq, rq = apply_exclusion_to_loader(test_loader_base, EXCLUDED_DATASET)

print(f"\n[EXCLUDE] Removed dataset='{EXCLUDED_DATASET}'")
print(f"  TRAIN: {bt} -> {at}  (removed {rt})")
print(f"  VAL:   {bv} -> {av}  (removed {rv})")
print(f"  TEST:  {bq} -> {aq}  (removed {rq})")

train_sequences_official = list(train_loader_base.dataset.sequences)
val_sequences_official   = list(val_loader_base.dataset.sequences)
test_sequences_official  = list(test_loader_base.dataset.sequences)

print("\n[OFFICIAL SEQUENCE COUNTS]")
print("  TRAIN:", len(train_sequences_official))
print("  VAL  :", len(val_sequences_official))
print("  TEST :", len(test_sequences_official))

print("\n[TRAIN split sample]")
print(train_sequences_official[0] if len(train_sequences_official) else None)

[REPO_ROOT] /RWKV-MedSAM2
[PKG_ROOT ] /RWKV-MedSAM2/rwkv_medsam2
[SAFE PACKAGE INSTALLED]
  rwkv_medsam2 in sys.modules: True
  rwkv_medsam2.dataset loaded : True
[PICKLE] top-level type: <class 'dict'>
[PICKLE] top-level keys: ['train_loaders', 'val_loaders', 'test_loaders']
[PICKLE] top-level loader-like keys: ['train_loaders', 'val_loaders', 'test_loaders']

[BASE LOADERS]
  Train batches: 153101
  Val batches  : 50989
  Test batches : 65866

[EXCLUDE] Before counts:
TRAIN unique datasets = 56 | excluded_count = 828
VAL unique datasets = 53 | excluded_count = 91
TEST unique datasets = 56 | excluded_count = 233

[EXCLUDE] Removed dataset='wanglab__CT_DeepLesion-MedSAM2'
  TRAIN: 459305 -> 458477  (removed 828)
  VAL:   50989 -> 50898  (removed 91)
  TEST:  65866 -> 65633  (removed 233)

[OFFICIAL SEQUENCE COUNTS]
  TRAIN: 458477
  VAL  : 50898
  TEST : 65633

[TRAIN split sample]
{'dataset': '2018-data-science-bowl', 'subdataset': 'default', 'tasks': ['cell_structures'], 'sequence': 

In [3]:
# %%
# =========================
# Cell 2 (NEW)
# Use the official train/val/test sequence pools directly
# =========================
from collections import Counter

def summarize_sequences(seqs, title):
    print(f"\n[{title}] n_sequences = {len(seqs)}")
    print("  dims      :", Counter(int(s.get("dim", -1)) for s in seqs))
    print("  modalities:", Counter(str(s.get("modality", "unknown")) for s in seqs).most_common(10))
    print("  datasets  :", Counter(str(s.get("dataset", "unknown")) for s in seqs).most_common(10))

train_sequences = train_sequences_official
val_sequences   = val_sequences_official
test_sequences  = test_sequences_official

summarize_sequences(train_sequences, "TRAIN (official)")
summarize_sequences(val_sequences,   "VAL   (official)")
summarize_sequences(test_sequences,  "TEST  (official)")

print("\n[INFO]")
print("  Refinement TRAIN will now use the SAME canonical training sequences as training.ipynb.")
print("  Refinement VAL   will now use the SAME canonical validation sequences as training.ipynb.")
print("  TEST remains untouched unless you explicitly build a test loader for evaluation.")


[TRAIN (official)] n_sequences = 458477
  dims      : Counter({2: 442503, 3: 15974})
  modalities: [('histopathology', 233353), ('x-ray', 174414), ('ultrasound', 15936), ('mri', 10840), ('ct', 10464), ('fetoscopy', 5255), ('retinal-scan', 3064), ('colonoscopy', 2794), ('dermoscopy', 2357)]
  datasets  : [('PAIP2019', 198937), ('CheXpert', 174243), ('2018-data-science-bowl', 19080), ('CoNSeP', 14001), ('ThyroidUltra', 12545), ('FetReg', 5255), ('Mosmeddata', 5003), ('AbdomenCT-1K', 3446), ('BRaTS2021', 2648), ('ISIC2018', 2357)]

[VAL   (official)] n_sequences = 50898
  dims      : Counter({2: 49150, 3: 1748})
  modalities: [('histopathology', 25925), ('x-ray', 19379), ('ultrasound', 1768), ('mri', 1193), ('ct', 1147), ('fetoscopy', 582), ('retinal-scan', 335), ('colonoscopy', 308), ('dermoscopy', 261)]
  datasets  : [('PAIP2019', 22104), ('CheXpert', 19360), ('2018-data-science-bowl', 2120), ('CoNSeP', 1555), ('ThyroidUltra', 1393), ('FetReg', 582), ('Mosmeddata', 555), ('AbdomenCT-1K

In [4]:
# %%
# =========================
# Cell 3 (REPLACEMENT)
# Oxford adapter dataset built directly from official training sequences
# SELF-CONTAINED: includes _ext and IMG_EXT_2D
# =========================
import os
import re
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset
import SimpleITK as sitk
from PIL import Image

def _ext(path):
    p = str(path).lower()
    if p.endswith(".nii.gz"):
        return ".nii.gz"
    return os.path.splitext(p)[1].lower()

IMG_EXT_2D = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def bbox_from_mask(mask_hw, pad=5):
    ys, xs = np.where(mask_hw > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    x0 = max(0, x0 - pad)
    y0 = max(0, y0 - pad)
    x1 = min(mask_hw.shape[1] - 1, x1 + pad)
    y1 = min(mask_hw.shape[0] - 1, y1 + pad)
    return [float(x0), float(y0), float(x1), float(y1)]

def resize_u8_gray(u8_hw, out_hw=512):
    t = torch.from_numpy(u8_hw).unsqueeze(0).unsqueeze(0).float()
    t = F.interpolate(t, size=(out_hw, out_hw), mode="bilinear", align_corners=False)
    return t.squeeze(0).squeeze(0).clamp(0, 255).byte().cpu().numpy()

def resize_mask_u8(mask_hw, out_hw=512):
    t = torch.from_numpy(mask_hw.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(out_hw, out_hw), mode="nearest")
    return (t.squeeze(0).squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)

def load_2d_gray_u8(path):
    ext = _ext(path)
    if ext in IMG_EXT_2D:
        im = Image.open(path).convert("L")
        return np.array(im, dtype=np.uint8)
    elif ext in [".nii", ".nii.gz"]:
        arr = sitk.GetArrayFromImage(sitk.ReadImage(path))
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(np.float32)
        lo, hi = np.percentile(arr, 0.5), np.percentile(arr, 99.5)
        if hi <= lo:
            hi = lo + 1.0
        arr = np.clip(arr, lo, hi)
        arr = (255.0 * (arr - lo) / (hi - lo)).clip(0, 255).astype(np.uint8)
        return arr
    elif ext == ".npy":
        arr = np.load(path)
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(np.float32)
        lo, hi = np.percentile(arr, 0.5), np.percentile(arr, 99.5)
        if hi <= lo:
            hi = lo + 1.0
        arr = np.clip(arr, lo, hi)
        arr = (255.0 * (arr - lo) / (hi - lo)).clip(0, 255).astype(np.uint8)
        return arr
    else:
        raise ValueError(f"Unsupported image ext for 2D load: {path}")

def load_2d_mask01(path):
    ext = _ext(path)
    if ext in IMG_EXT_2D:
        m = Image.open(path).convert("L")
        return (np.array(m) > 0).astype(np.uint8)
    elif ext in [".nii", ".nii.gz"]:
        arr = sitk.GetArrayFromImage(sitk.ReadImage(path))
        if arr.ndim == 3:
            arr = arr[0]
        return (arr > 0).astype(np.uint8)
    elif ext == ".npy":
        arr = np.load(path)
        if arr.ndim == 3:
            arr = arr[0]
        return (arr > 0).astype(np.uint8)
    else:
        raise ValueError(f"Unsupported mask ext for 2D load: {path}")

def _extract_frame_pairs(seqrec):
    """
    Robustly extract list[(img_path, mask_path)] from official sequence records.
    Supports:
      - sequence = [(img, mask), ...]
      - sequence = [{"image":..., "mask":...}, ...]
      - top-level image/mask fallback
    """
    seq = seqrec.get("sequence", None)

    if isinstance(seq, list) and len(seq) > 0:
        out = []
        for item in seq:
            if isinstance(item, (tuple, list)) and len(item) >= 2:
                out.append((str(item[0]), str(item[1])))
            elif isinstance(item, dict):
                img = item.get("image", None)
                msk = item.get("mask", None)
                if img is not None and msk is not None:
                    out.append((str(img), str(msk)))
        if len(out) > 0:
            return out

    if seqrec.get("image") is not None and seqrec.get("mask") is not None:
        return [(str(seqrec["image"]), str(seqrec["mask"]))]

    return []

SEQ_LEN_3D = 8

class OxfordFromTrainingSequences(Dataset):
    def __init__(self, sequences, target_hw=512, pad=5, seq_len_3d=8, is_val=False, seed=42):
        self.sequences = sequences
        self.target_hw = int(target_hw)
        self.pad = int(pad)
        self.seq_len_3d = int(seq_len_3d)
        self.is_val = bool(is_val)
        self.seed = int(seed)

    def __len__(self):
        return len(self.sequences)

    def _rng_for_index(self, idx):
        s = (self.seed * 1000003) ^ (int(idx) * 9176) ^ 0x9E3779B1
        s = int(s % (2**32 - 1))
        return np.random.RandomState(s)

    def __getitem__(self, idx):
        task_id = None
        if isinstance(idx, (tuple, list)) and len(idx) == 2:
            idx, task_id = idx

        idx = int(idx)
        rec = self.sequences[idx]

        if task_id is None:
            tasks = rec.get("tasks", [])
            if isinstance(tasks, (list, tuple)) and len(tasks) > 0:
                task_id = tasks[0]
            elif tasks is not None:
                task_id = tasks
        if task_id is not None:
            task_id = str(task_id)

        dim = int(rec.get("dim", 2))
        frame_pairs = _extract_frame_pairs(rec)

        if len(frame_pairs) == 0:
            return None

        if dim == 2:
            img_path, msk_path = frame_pairs[0]
            if (not os.path.exists(img_path)) or (not os.path.exists(msk_path)):
                return None

            img_u8 = load_2d_gray_u8(img_path)
            msk_u8 = load_2d_mask01(msk_path)

            img_r = resize_u8_gray(img_u8, self.target_hw)
            msk_r = resize_mask_u8(msk_u8, self.target_hw)

            box = bbox_from_mask(msk_r, pad=self.pad)
            if box is None:
                return None

            img_rgb = np.repeat(img_r[..., None], 3, axis=2)
            img_t = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
            msk_t = torch.from_numpy(msk_r).unsqueeze(0).float()
            box_t = torch.tensor(box, dtype=torch.float32)

            imgs = img_t.unsqueeze(0)
            msks = msk_t.unsqueeze(0)
            boxs = box_t.unsqueeze(0)

            image_ref = img_path
            mask_ref = msk_path

        else:
            valid_frames = []
            for img_path, msk_path in frame_pairs:
                if (not os.path.exists(img_path)) or (not os.path.exists(msk_path)):
                    continue
                try:
                    msk_u8 = load_2d_mask01(msk_path)
                except Exception:
                    continue
                if msk_u8.sum() > 0:
                    valid_frames.append((img_path, msk_path))

            if len(valid_frames) == 0:
                return None

            T = self.seq_len_3d
            rng = self._rng_for_index(idx) if self.is_val else np.random

            frame_indices = np.arange(len(valid_frames))
            if len(valid_frames) >= T:
                chosen_idx = rng.choice(frame_indices, size=T, replace=False)
            else:
                chosen_idx = rng.choice(frame_indices, size=T, replace=True)

            chosen_idx = np.sort(chosen_idx).astype(int)
            chosen_pairs = [valid_frames[i] for i in chosen_idx]

            imgs_list, msks_list, box_list = [], [], []
            for img_path, msk_path in chosen_pairs:
                img_u8 = load_2d_gray_u8(img_path)
                msk_u8 = load_2d_mask01(msk_path)

                img_r = resize_u8_gray(img_u8, self.target_hw)
                msk_r = resize_mask_u8(msk_u8, self.target_hw)

                box = bbox_from_mask(msk_r, pad=self.pad)
                if box is None:
                    return None

                img_rgb = np.repeat(img_r[..., None], 3, axis=2)
                img_t = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
                msk_t = torch.from_numpy(msk_r).unsqueeze(0).float()
                box_t = torch.tensor(box, dtype=torch.float32)

                imgs_list.append(img_t)
                msks_list.append(msk_t)
                box_list.append(box_t)

            imgs = torch.stack(imgs_list, 0)
            msks = torch.stack(msks_list, 0)
            boxs = torch.stack(box_list, 0)

            image_ref = chosen_pairs[0][0]
            mask_ref = chosen_pairs[0][1]

        meta = {
            "dataset": rec.get("dataset"),
            "subdataset": rec.get("subdataset"),
            "modality": rec.get("modality"),
            "pipeline": rec.get("pipeline", None),
            "identifier": rec.get("identifier", None),
            "class": rec.get("class", None),
            "dim": dim,
            "image": image_ref,
            "mask": mask_ref,
            "pair_index": idx,
            "task_id": task_id,
            "task_label": rec.get("task_label", None),
        }
        return imgs, msks, boxs, meta

def collate_pad_none_seq(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None

    imgs_list, msks_list, boxs_list, metas = zip(*batch)

    T_list = [x.shape[0] for x in imgs_list]
    T_max = max(T_list)

    imgs_pad, msks_pad, boxs_pad, valids = [], [], [], []
    for imgs, msks, boxs, T in zip(imgs_list, msks_list, boxs_list, T_list):
        if T < T_max:
            pad_n = T_max - T
            imgs = torch.cat([imgs, imgs[-1:].repeat(pad_n, 1, 1, 1)], dim=0)
            msks = torch.cat([msks, msks[-1:].repeat(pad_n, 1, 1, 1)], dim=0)
            boxs = torch.cat([boxs, boxs[-1:].repeat(pad_n, 1)], dim=0)

        valid = torch.zeros(T_max, dtype=torch.bool)
        valid[:T] = True

        imgs_pad.append(imgs)
        msks_pad.append(msks)
        boxs_pad.append(boxs)
        valids.append(valid)

    return (
        torch.stack(imgs_pad, 0),
        torch.stack(msks_pad, 0),
        torch.stack(boxs_pad, 0),
        torch.stack(valids, 0),
        list(metas),
    )

In [5]:
# %%
# =========================
# Cell 4 (REPLACEMENT)
# Build Oxford refinement loaders from the EXACT official sequence pools
# but with FRESH samplers so indices match the Oxford dataset lengths
# =========================
import torch
from torch.utils.data import DataLoader

print("[OFFICIAL SEQUENCE POOLS]")
print("  TRAIN sequences:", len(train_sequences))
print("  VAL sequences  :", len(val_sequences))
print("  TEST sequences :", len(test_sequences))

# -------------------------
# Oxford adapter datasets built from official sequence pools
# -------------------------
train_ds = OxfordFromTrainingSequences(
    train_sequences,
    target_hw=TARGET_HW,
    pad=5,
    seq_len_3d=SEQ_LEN_3D,
    is_val=False,
    seed=SEED,
)
val_ds = OxfordFromTrainingSequences(
    val_sequences,
    target_hw=TARGET_HW,
    pad=5,
    seq_len_3d=SEQ_LEN_3D,
    is_val=True,
    seed=SEED,
)
test_ds = OxfordFromTrainingSequences(
    test_sequences,
    target_hw=TARGET_HW,
    pad=5,
    seq_len_3d=SEQ_LEN_3D,
    is_val=True,
    seed=SEED,
)

# -------------------------
# Helpers to rebuild samplers from the official sequence pools
# -------------------------
def build_tasks_map_from_sequences(seqs):
    tasks_map = {}
    for rec in seqs:
        ds = rec.get("dataset", "unknown")
        sd = rec.get("subdataset", "default")

        task = rec.get("task_id", None) or rec.get("tasks", None) or rec.get("task", None)
        if isinstance(task, (list, tuple)):
            task = task[0] if len(task) > 0 else None
        elif isinstance(task, dict):
            task = task.get("id", None) or task.get("task_id", None) or task.get("name", None)

        if task is None:
            task = f"{ds}/{sd}"
        task = str(task)

        if task not in tasks_map:
            tasks_map[task] = {"classes": [], "datasets": {}}

        tasks_map[task]["datasets"].setdefault(ds, set()).add(sd)

    for t in list(tasks_map.keys()):
        tasks_map[t]["datasets"] = {k: sorted(list(v)) for k, v in tasks_map[t]["datasets"].items()}

    return tasks_map

tasks_map_train = build_tasks_map_from_sequences(train_sequences)
tasks_map_val   = build_tasks_map_from_sequences(val_sequences)
tasks_map_test  = build_tasks_map_from_sequences(test_sequences)

# -------------------------
# Use the SAME batch size as training.ipynb if available
# -------------------------
BATCH_SIZE = getattr(train_loader_base, "batch_size", None)
if BATCH_SIZE is None:
    try:
        BATCH_SIZE = int(cfg.training.batch_size)
    except Exception:
        BATCH_SIZE = 3

print("[SAMPLER CONFIG]")
print("  batch_size:", BATCH_SIZE)

# -------------------------
# Fresh train sampler
# -------------------------
def make_train_sampler(epoch: int):
    return BalancedTaskBatchSampler(
        sequences=train_sequences,
        tasks_map=tasks_map_train,
        batch_size=BATCH_SIZE,
        drop_last=True,
        seed=SEED + int(epoch),
        min_per_bucket=1,
    )

# -------------------------
# Fresh full validation sampler
# -------------------------
def make_val_sampler():
    return BalancedTaskBatchSampler(
        sequences=val_sequences,
        tasks_map=tasks_map_val,
        batch_size=1,      # keep validation item-wise, like before
        drop_last=False,
        seed=SEED,
        min_per_bucket=1,
    )

# -------------------------
# Optional test sampler
# -------------------------
def make_test_sampler():
    return BalancedTaskBatchSampler(
        sequences=test_sequences,
        tasks_map=tasks_map_test,
        batch_size=1,
        drop_last=False,
        seed=SEED,
        min_per_bucket=1,
    )

# Freeze validation/test batches once for determinism
frozen_val_batches = list(make_val_sampler())
frozen_test_batches = list(make_test_sampler())

print("[VAL/FULL] batches :", len(frozen_val_batches))
print("[TEST/FULL] batches:", len(frozen_test_batches))

class FrozenBatchSampler(torch.utils.data.Sampler):
    def __init__(self, frozen_batches):
        self.frozen_batches = [list(b) for b in frozen_batches]
    def __iter__(self):
        for b in self.frozen_batches:
            yield b
    def __len__(self):
        return len(self.frozen_batches)

val_batch_sampler = FrozenBatchSampler(frozen_val_batches)
test_batch_sampler = FrozenBatchSampler(frozen_test_batches)

val_loader = DataLoader(
    val_ds,
    batch_sampler=val_batch_sampler,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_pad_none_seq,
    persistent_workers=False,
)

test_loader = DataLoader(
    test_ds,
    batch_sampler=test_batch_sampler,
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_pad_none_seq,
    persistent_workers=False,
)

def make_train_loader(epoch: int):
    train_batch_sampler = make_train_sampler(epoch)
    return DataLoader(
        train_ds,
        batch_sampler=train_batch_sampler,
        num_workers=0,
        pin_memory=False,
        collate_fn=collate_pad_none_seq,
        persistent_workers=False,
    )

print("\n[OXFORD LOADERS FROM OFFICIAL SEQUENCES]")
print("  train_ds items:", len(train_ds))
print("  val_ds items  :", len(val_ds))
print("  test_ds items :", len(test_ds))
print("  val batches   :", len(val_loader))
print("  test batches  :", len(test_loader))

# Smoke batch
vb = next(iter(val_loader))
imgs, msks, boxes, valid, metas = vb
print("\n[VAL BATCH]")
print("  imgs :", imgs.shape)
print("  msks :", msks.shape)
print("  boxes:", boxes.shape)
print("  valid:", valid.shape)
print("  meta :", metas[0] if len(metas) else None)

[OFFICIAL SEQUENCE POOLS]
  TRAIN sequences: 458477
  VAL sequences  : 50898
  TEST sequences : 65633
[SAMPLER CONFIG]
  batch_size: 3
[VAL/FULL] batches : 50645
[TEST/FULL] batches: 64811

[OXFORD LOADERS FROM OFFICIAL SEQUENCES]
  train_ds items: 458477
  val_ds items  : 50898
  test_ds items : 65633
  val batches   : 50645
  test batches  : 64811

[VAL BATCH]
  imgs : torch.Size([1, 1, 3, 512, 512])
  msks : torch.Size([1, 1, 1, 512, 512])
  boxes: torch.Size([1, 1, 4])
  valid: torch.Size([1, 1])
  meta : {'dataset': 'FetReg', 'subdataset': 'default', 'modality': 'fetoscopy', 'pipeline': None, 'identifier': None, 'class': None, 'dim': 2, 'image': '/data/Preprocessed/FetReg/Fetoscopy/default/train/Video003/frame15270/images/db6e8cd9_img000.png', 'mask': '/data/Preprocessed/FetReg/Fetoscopy/default/train/Video003/frame15270/masks/db6e8cd9_img000_mask000_%fetus%_comp002.png', 'pair_index': 23351, 'task_id': 'fetus', 'task_label': None}


In [6]:
# %%
# %%
# =========================
# Cell 5 — Oxford model setup (UPDATED):
#   - Build SAM2 VideoPredictor natively at 512 via Hydra overrides
#   - Load MedSAM2_pretrain.pth
# =========================
import os, sys, glob, inspect
import torch
from hydra.core.global_hydra import GlobalHydra

BASE = "/data/jrbonill/oxford_medsam2_env"
SITE = f"{BASE}/site"
REPO = f"{BASE}/Medical-SAM2"
CKPT_DIR = f"{REPO}/checkpoints"

for p in [REPO, SITE]:
    if p not in sys.path:
        sys.path.insert(0, p)

SAM_CKPT     = f"{CKPT_DIR}/sam2_hiera_tiny.pt"      # choose what you want
MED_PRETRAIN = f"{CKPT_DIR}/MedSAM2_pretrain.pth"

assert os.path.exists(SAM_CKPT), SAM_CKPT
assert os.path.exists(MED_PRETRAIN), MED_PRETRAIN

# Clear Hydra BEFORE importing sam2_train (sam2_train auto-inits Hydra on import)
GlobalHydra.instance().clear()

from sam2_train.build_sam import build_sam2_video_predictor

SAM_CONFIG_NAME = "sam2_hiera_t"  # name, not path

# ---------------------------------------------------------
# CHANGE (1): Build predictor natively at 512 (no 1024↔512)
# ---------------------------------------------------------
TARGET_IMAGE_SIZE = int(TARGET_HW)  # 512

# For 1024 configs, feat_sizes usually [32,32]. For 512 -> [16,16].
hydra_overrides_extra = [
    f"model.image_size={TARGET_IMAGE_SIZE}",
    "model.memory_attention.layer.self_attention.feat_sizes=[16,16]",
    "model.memory_attention.layer.cross_attention.feat_sizes=[16,16]",
]

predictor = build_sam2_video_predictor(
    config_file=SAM_CONFIG_NAME,
    ckpt_path=SAM_CKPT,
    device="cuda",
    mode="train",
    hydra_overrides_extra=hydra_overrides_extra,
)

# --- Disable hole filling to avoid sam2_train/_C.so connected-components extension ---
if hasattr(predictor, "fill_hole_area"):
    predictor.fill_hole_area = 0
if hasattr(predictor, "model") and hasattr(predictor.model, "fill_hole_area"):
    predictor.model.fill_hole_area = 0

print("predictor.fill_hole_area =", getattr(predictor, "fill_hole_area", None))
print("predictor.model.fill_hole_area =", getattr(getattr(predictor, "model", None), "fill_hole_area", None))

# Helper to infer predictor expected size
def _predictor_target_hw(predictor):
    for attr in ["image_size", "img_size", "target_size"]:
        if hasattr(predictor, attr):
            v = getattr(predictor, attr)
            if isinstance(v, int) and v > 0:
                return int(v), int(v)

    m = getattr(predictor, "model", None)
    if m is not None:
        for attr in ["image_size", "img_size", "target_size"]:
            if hasattr(m, attr):
                v = getattr(m, attr)
                if isinstance(v, int) and v > 0:
                    return int(v), int(v)

        if hasattr(m, "sam_image_embedding_size"):
            emb = int(getattr(m, "sam_image_embedding_size"))
            return emb * 16, emb * 16

    return int(TARGET_HW), int(TARGET_HW)

print("Requested TARGET_IMAGE_SIZE:", TARGET_IMAGE_SIZE)
print("Predictor inferred size    :", _predictor_target_hw(predictor))

assert _predictor_target_hw(predictor) == (TARGET_IMAGE_SIZE, TARGET_IMAGE_SIZE), (
    f"Predictor is not native {TARGET_IMAGE_SIZE}. "
    f"Got {_predictor_target_hw(predictor)}. "
    "Hydra overrides may not have applied."
)

# ---------------------------------------------------------
# Load MedSAM2_pretrain.pth into the underlying model module
# ---------------------------------------------------------
def unwrap_state_dict(obj):
    if isinstance(obj, dict):
        for k in ["state_dict", "model", "model_state_dict", "net", "network"]:
            if k in obj and isinstance(obj[k], dict):
                return obj[k]
    return obj

def get_module_for_loading(p):
    if isinstance(p, torch.nn.Module):
        return p
    for attr in ["model", "sam2", "sam", "net", "network"]:
        if hasattr(p, attr) and isinstance(getattr(p, attr), torch.nn.Module):
            return getattr(p, attr)
    for _, v in vars(p).items():
        if isinstance(v, torch.nn.Module):
            return v
    raise RuntimeError("Could not locate a torch.nn.Module inside predictor.")

model = get_module_for_loading(predictor).to(DEVICE).train()

ckpt_obj = torch.load(MED_PRETRAIN, map_location="cpu")
state = unwrap_state_dict(ckpt_obj)
missing, unexpected = model.load_state_dict(state, strict=False)

print("Loaded MED_PRETRAIN (strict=False)")
print("  Missing keys   :", len(missing))
print("  Unexpected keys:", len(unexpected))
print("Model type:", type(model))


predictor.fill_hole_area = 0
predictor.model.fill_hole_area = None
Requested TARGET_IMAGE_SIZE: 512
Predictor inferred size    : (512, 512)
Loaded MED_PRETRAIN (strict=False)
  Missing keys   : 0
  Unexpected keys: 0
Model type: <class 'sam2_train.sam2_video_predictor.SAM2VideoPredictor'>


In [7]:
# =========================
# Cell 6 (REPLACEMENT)
# UPDATED:
#   - no separate mask_prompts tensor
#   - when prompt_mode == "mask", reuse msks directly
#   - TRAIN: 50% box / 50% mask
#   - VAL:   100% box
# =========================
import torch
import torch.nn.functional as F
import numpy as np

TRAIN_PROMPT_TYPE_PROBS = {
    "box": 0.5,
    "mask": 0.5,
}

VAL_PROMPT_TYPE_PROBS = {
    "box": 1.0,
    "mask": 0.0,
}

def forward_oxford_prompt(model, imgs_b3hw_01, boxes_b4=None, mask_prompts_b1hw=None, prompt_mode="box"):
    B, _, H, W = imgs_b3hw_01.shape

    backbone_out = model.forward_image(imgs_b3hw_01)
    _, vision_feats, vision_pos_embeds, feat_sizes = model._prepare_backbone_features(backbone_out)

    feats = [
        feat.permute(1, 2, 0).reshape(B, -1, *sz)
        for feat, sz in zip(vision_feats[::-1], feat_sizes[::-1])
    ][::-1]
    image_embed = feats[-1]
    hires_feats = feats[:-1]

    if prompt_mode == "box":
        boxes = boxes_b4.view(B, 2, 2)
        sparse, dense = model.sam_prompt_encoder(points=None, boxes=boxes, masks=None)
    elif prompt_mode == "mask":
        sparse, dense = model.sam_prompt_encoder(points=None, boxes=None, masks=mask_prompts_b1hw)
    else:
        raise ValueError(f"Unsupported prompt_mode: {prompt_mode}")

    if dense.shape[-2:] != image_embed.shape[-2:]:
        dense = F.interpolate(dense, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)

    image_pe = model.sam_prompt_encoder.get_dense_pe()
    if image_pe.shape[-2:] != image_embed.shape[-2:]:
        image_pe = F.interpolate(image_pe, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)
    image_pe = image_pe.to(image_embed.dtype)

    lowres_logits, _, _, _ = model.sam_mask_decoder(
        image_embeddings=image_embed,
        image_pe=image_pe,
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        multimask_output=False,
        repeat_image=False,
        high_res_features=hires_feats,
    )

    logits = F.interpolate(lowres_logits, size=(H, W), mode="bilinear", align_corners=False)
    return logits

def _predictor_target_hw(predictor):
    for attr in ["image_size", "img_size", "target_size"]:
        if hasattr(predictor, attr):
            v = getattr(predictor, attr)
            if isinstance(v, int) and v > 0:
                return int(v), int(v)

    m = getattr(predictor, "model", None)
    if m is not None:
        for attr in ["image_size", "img_size", "target_size"]:
            if hasattr(m, attr):
                v = getattr(m, attr)
                if isinstance(v, int) and v > 0:
                    return int(v), int(v)

        if hasattr(m, "sam_image_embedding_size"):
            emb = int(getattr(m, "sam_image_embedding_size"))
            return emb * 16, emb * 16

    return int(TARGET_HW), int(TARGET_HW)

def _seeded_rng_for_val(epoch: int, meta: dict):
    base = int(globals().get("SEED", 42))
    pair_index = meta.get("pair_index", None)
    try:
        pair_index = int(pair_index) if pair_index is not None else 0
    except Exception:
        pair_index = 0
    s = (base * 1000003) ^ (int(epoch) * 1009) ^ (pair_index * 9176)
    s = int(s % (2**32 - 1))
    return np.random.RandomState(s)

def _sample_prompt_mode_train():
    return np.random.choice(
        ["box", "mask"],
        p=[TRAIN_PROMPT_TYPE_PROBS["box"], TRAIN_PROMPT_TYPE_PROBS["mask"]],
    )

def _sample_prompt_mode_val(epoch: int, meta: dict):
    rng = _seeded_rng_for_val(epoch, meta)
    return rng.choice(
        ["box", "mask"],
        p=[VAL_PROMPT_TYPE_PROBS["box"], VAL_PROMPT_TYPE_PROBS["mask"]],
    ).item()

def _choose_prompt_frames_default(seq_valid_t, seq_nonempty_t):
    cand = torch.nonzero(seq_valid_t & seq_nonempty_t, as_tuple=False).squeeze(1)
    if cand.numel() == 0:
        return []
    return [int(cand[0].item())]

def _choose_prompt_frames_probabilistic(seq_valid_t, seq_nonempty_t, prompt_prob: float, rng: np.random.RandomState):
    cand = torch.nonzero(seq_valid_t & seq_nonempty_t, as_tuple=False).squeeze(1)
    if cand.numel() == 0:
        return []

    cand_np = cand.detach().cpu().numpy().astype(int)
    keep = rng.rand(cand_np.shape[0]) < float(prompt_prob)
    chosen = cand_np[keep]

    if chosen.size == 0:
        chosen = np.array([cand_np[rng.randint(0, cand_np.shape[0])]], dtype=int)

    chosen = np.unique(chosen)
    chosen.sort()
    return [int(x) for x in chosen.tolist()]

def forward_oxford_video_predictor_sequence(
    predictor,
    imgs_bt3hw_01,
    boxes_bt4,
    msks_bt1hw,
    valid_bt=None,
    nonempty_bt=None,
    obj_id=1,
    prompt_frames_by_b=None,
    prompt_mode_by_b=None,
):
    assert imgs_bt3hw_01.ndim == 5, f"Expected [B,T,3,H,W], got {imgs_bt3hw_01.shape}"
    B, T, C, H0, W0 = imgs_bt3hw_01.shape
    assert C == 3, f"Expected 3 channels, got {C}"

    if nonempty_bt is None:
        raise ValueError("nonempty_bt is required to avoid prompting empty frames.")

    Ht, Wt = _predictor_target_hw(predictor)

    imgs = imgs_bt3hw_01
    boxes = boxes_bt4
    msks = msks_bt1hw

    if (H0, W0) != (Ht, Wt):
        imgs = imgs.reshape(B * T, 3, H0, W0)
        imgs = F.interpolate(imgs, size=(Ht, Wt), mode="bilinear", align_corners=False)
        imgs = imgs.reshape(B, T, 3, Ht, Wt)

        msk_resized = msks.reshape(B * T, 1, H0, W0).float()
        msk_resized = F.interpolate(msk_resized, size=(Ht, Wt), mode="nearest")
        msks = msk_resized.reshape(B, T, 1, Ht, Wt)

        sx = float(Wt) / float(W0)
        sy = float(Ht) / float(H0)
        boxes = boxes.clone()
        boxes[..., 0] *= sx
        boxes[..., 2] *= sx
        boxes[..., 1] *= sy
        boxes[..., 3] *= sy

    if valid_bt is None:
        valid_bt = torch.ones((B, T), dtype=torch.bool, device=imgs.device)
    else:
        valid_bt = valid_bt.to(device=imgs.device).bool()

    nonempty_bt = nonempty_bt.to(device=imgs.device).bool()

    logits_pred = torch.zeros((B, T, 1, Ht, Wt), dtype=imgs.dtype, device=imgs.device)

    for b in range(B):
        seq_imgs  = imgs[b]
        seq_box   = boxes[b]
        seq_msk   = msks[b]
        seq_valid = valid_bt[b]
        seq_ne    = nonempty_bt[b]

        if int((seq_valid & seq_ne).sum().item()) == 0:
            continue

        state = predictor.train_init_state(imgs_tensor=seq_imgs)

        if prompt_frames_by_b is not None:
            prompt_frames = list(prompt_frames_by_b[b]) if (b < len(prompt_frames_by_b)) else []
            prompt_frames = [int(f) for f in prompt_frames if (0 <= int(f) < T) and bool((seq_valid[int(f)] & seq_ne[int(f)]).item())]
            if len(prompt_frames) == 0:
                prompt_frames = _choose_prompt_frames_default(seq_valid, seq_ne)
        else:
            prompt_frames = _choose_prompt_frames_default(seq_valid, seq_ne)

        prompt_mode = prompt_mode_by_b[b] if (prompt_mode_by_b is not None and b < len(prompt_mode_by_b)) else "box"

        for fid in prompt_frames:
            if prompt_mode == "box":
                predictor.train_add_new_bbox(
                    inference_state=state,
                    frame_idx=int(fid),
                    obj_id=int(obj_id),
                    bbox=seq_box[int(fid)],
                    clear_old_points=False,
                )
            elif prompt_mode == "mask":
                predictor.train_add_new_mask(
                    inference_state=state,
                    frame_idx=int(fid),
                    obj_id=int(obj_id),
                    mask=seq_msk[int(fid), 0],
                )
            else:
                raise ValueError(f"Unsupported prompt_mode: {prompt_mode}")

        video_segments = {}
        for out_fid, out_obj_ids, out_mask_logits in predictor.train_propagate_in_video(state, start_frame_idx=0):
            video_segments[int(out_fid)] = {
                int(oid): out_mask_logits[i] for i, oid in enumerate(out_obj_ids)
            }

        for fid in range(T):
            if not bool(seq_valid[fid].item()):
                continue
            if fid not in video_segments:
                continue
            if int(obj_id) not in video_segments[fid]:
                continue

            x = video_segments[fid][int(obj_id)]
            if x.ndim == 2:
                x = x.unsqueeze(0)
            elif x.ndim == 3 and x.shape[0] != 1:
                x = x[:1]
            logits_pred[b, fid] = x

        predictor.reset_state(state)

    if (Ht, Wt) != (H0, W0):
        logits_out = logits_pred.reshape(B * T, 1, Ht, Wt)
        logits_out = F.interpolate(logits_out, size=(H0, W0), mode="bilinear", align_corners=False)
        logits_out = logits_out.reshape(B, T, 1, H0, W0)
        return logits_out

    return logits_pred

In [9]:
# %%
# =========================
# Cell 7A (REPLACEMENT)
# Scheduler/logging sized from the rebuilt train loader
# =========================
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

%load_ext tensorboard

EPOCHS = 20
MAX_VAL_BATCHES = None
LOG_EVERY_STEPS = 25
PRINT_VAL_EVERY_EPOCH = True
TEXT_PROGRESS_EVERY_STEPS = 100

def freeze_for_teacher_refine(model):
    for p in model.parameters():
        p.requires_grad = False

    train_prefixes = [
        "sam_prompt_encoder.",
        "sam_mask_decoder.",
    ]

    n_train = 0
    n_total = 0
    for name, p in model.named_parameters():
        n_total += p.numel()
        if any(name.startswith(pref) for pref in train_prefixes):
            p.requires_grad = True
            n_train += p.numel()

    print("[FREEZE] total params:", f"{n_total:,}")
    print("[FREEZE] trainable params:", f"{n_train:,}")
    print("[FREEZE] trainable ratio:", f"{(n_train / max(1, n_total)):.6f}")

freeze_for_teacher_refine(model)

prompt_lr = 5e-6
decoder_lr = 1e-5
wd = 0.01

prompt_params = []
decoder_params = []
other_trainable = []

for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    if name.startswith("sam_prompt_encoder."):
        prompt_params.append(p)
    elif name.startswith("sam_mask_decoder."):
        decoder_params.append(p)
    else:
        other_trainable.append(p)

if other_trainable:
    print("[WARN] Unexpected trainable params outside prompt/mask decoder:", len(other_trainable))

optimizer = torch.optim.AdamW(
    [
        {"params": prompt_params, "lr": float(prompt_lr), "weight_decay": float(wd)},
        {"params": decoder_params, "lr": float(decoder_lr), "weight_decay": float(wd)},
    ]
)

_tmp_train_loader = make_train_loader(epoch=0)
STEPS_PER_EPOCH = len(_tmp_train_loader)
del _tmp_train_loader

TOTAL_STEPS = EPOCHS * STEPS_PER_EPOCH
warmup_steps = max(1, int(0.1 * TOTAL_STEPS))

print("[SCHED] steps/epoch =", STEPS_PER_EPOCH)
print("[SCHED] total_steps  =", TOTAL_STEPS)
print("[SCHED] warmup_steps =", warmup_steps)
print("[VAL] full validation batches =", len(val_loader))

def lr_lambda(current_step):
    if current_step < warmup_steps:
        return float(current_step + 1) / float(warmup_steps)
    progress = float(current_step - warmup_steps) / float(max(1, TOTAL_STEPS - warmup_steps))
    return max(0.0, 0.5 * (1.0 + math.cos(math.pi * progress)))

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

tb_dir = os.path.join(OUT_ROOT, "tb")
run_dir = os.path.join(
    tb_dir,
    datetime.now().strftime("%Y%m%d-%H%M%S") + f"-seed{SEED}-bs{BATCH_SIZE}"
)

writer = SummaryWriter(log_dir=run_dir, flush_secs=5, max_queue=10)

writer.add_hparams(
    {
        "bs": int(BATCH_SIZE) if BATCH_SIZE is not None else -1,
        "epochs": int(EPOCHS),
        "prompt_lr": float(prompt_lr),
        "decoder_lr": float(decoder_lr),
        "weight_decay": float(wd),
        "seed": int(SEED),
        "steps_per_epoch": int(STEPS_PER_EPOCH),
        "total_steps": int(TOTAL_STEPS),
        "val_batches": int(len(val_loader)),
        "seq_len_3d": int(SEQ_LEN_3D),
        "train_prompt_mode_mix": "box=0.5,mask=0.5",
        "val_prompt_mode_mix": "box=1.0,mask=0.0",
        "data_source": "official_loaders32.pkl",
        "seq_policy_3d": "official_sequences__random_8_nonempty_train__deterministic_8_nonempty_val",
    },
    {"val/iou_best": 0.0}
)

print("TensorBoard run_dir:", run_dir)

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard
[FREEZE] total params: 38,945,986
[FREEZE] trainable params: 4,221,329
[FREEZE] trainable ratio: 0.108389
[SCHED] steps/epoch = 152825
[SCHED] total_steps  = 3056500
[SCHED] warmup_steps = 305650
[VAL] full validation batches = 50645
TensorBoard run_dir: /data/jrbonill/oxford_refine_runs/oxford_refine_20260312_023627/tb/20260312-024126-seed42-bs3


In [10]:
# %%
import subprocess, os, time

PORT = 6007
HOST = "127.0.0.1"

# kill old tensorboards on this port (best-effort)
subprocess.run(["bash", "-lc", f"pkill -f 'tensorboard.*--port {PORT}' || true"], check=False)

proc = subprocess.Popen(
    ["tensorboard", "--logdir", run_dir, "--port", str(PORT), "--host", HOST, "--reload_multifile", "true"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

print("TensorBoard PID:", proc.pid)
print(f"Now serving on http://{HOST}:{PORT}")
time.sleep(1)

TensorBoard PID: 3551570
Now serving on http://127.0.0.1:6007


In [11]:
# %%
from IPython.display import IFrame, display

display(IFrame(src="http://127.0.0.1:6007", width="100%", height=900))

In [ ]:
# =========================
# Cell 7B (REPLACEMENT)
# UPDATED:
#   - full-data training compatible
#   - TensorBoard text progress updates
# =========================
from tqdm.auto import tqdm
import numpy as np
import time
import html
import re
import os
import json
import torch
import torch.nn.functional as F

PROMPT_FRAME_PROB_2D = 1.0
PROMPT_FRAME_PROB_3D = 0.25

VIS_EVERY_EPOCHS = 5
VIS_MAX_BATCHES  = 1
VIS_MAX_ITEMS    = 1
VIS_FPS          = 6
VIS_PING_PONG    = True

VAL_EVERY_EPOCHS = 5

AMP_ENABLED = torch.cuda.is_available()
AMP_DTYPE   = torch.float16 if AMP_ENABLED else None

def _load_compute_metrics_funcs():
    if "compute_metrics_2d" in globals() and "compute_metrics_3d" in globals():
        return globals()["compute_metrics_2d"], globals()["compute_metrics_3d"]
    raise RuntimeError(
        "compute_metrics_2d / compute_metrics_3d wrappers are not defined. "
        "Re-run Cell 0 so the notebook can rebuild them from the current func_metrics.py helpers."
    )

compute_metrics_2d, compute_metrics_3d = _load_compute_metrics_funcs()

def tb_pre(text: str) -> str:
    if text is None:
        text = ""
    return "<pre style='font-family: monospace; white-space: pre;'>" + html.escape(str(text)) + "</pre>"

def get_lr_group(opt, group_idx=0):
    if not hasattr(opt, "param_groups") or len(opt.param_groups) == 0:
        return float("nan")
    group_idx = min(max(0, group_idx), len(opt.param_groups) - 1)
    return float(opt.param_groups[group_idx].get("lr", float("nan")))

def bce_dice_loss(logits, gt):
    bce = F.binary_cross_entropy_with_logits(logits, gt, reduction="mean")
    probs = torch.sigmoid(logits)
    inter = (probs * gt).flatten(1).sum(1)
    denom = probs.flatten(1).sum(1) + gt.flatten(1).sum(1) + 1e-6
    dice = 1.0 - (2 * inter / denom).mean()
    return bce + dice

def dice_torch_from_logits(logits_b1hw, gt_b1hw):
    probs = torch.sigmoid(logits_b1hw)
    pred = (probs > 0.5).float()
    inter = (pred * gt_b1hw).flatten(1).sum(1)
    denom = pred.flatten(1).sum(1) + gt_b1hw.flatten(1).sum(1) + 1e-6
    return (2 * inter / denom).mean()

@torch.no_grad()
def iou_torch_from_logits(logits_b1hw, gt_b1hw, thr=0.5):
    probs = torch.sigmoid(logits_b1hw)
    pred = (probs > thr).float()
    inter = (pred * gt_b1hw).flatten(1).sum(1)
    union = (pred + gt_b1hw - pred * gt_b1hw).flatten(1).sum(1) + 1e-6
    return (inter / union).mean()

def _select_valid_frames(flat_logits, flat_msks, flat_valid_bool):
    if flat_valid_bool is None:
        return flat_logits, flat_msks
    flat_valid_bool = flat_valid_bool.to(device=flat_logits.device)
    idx = torch.nonzero(flat_valid_bool, as_tuple=False).squeeze(1)
    if idx.numel() == 0:
        return None, None
    return flat_logits.index_select(0, idx), flat_msks.index_select(0, idx)

def _safe_tag(s: str) -> str:
    s = str(s) if s is not None else "unknown"
    s = re.sub(r"\s+", "_", s.strip())
    s = re.sub(r"[^A-Za-z0-9_\-\.]+", "_", s)
    return s[:120]

def _build_prompt_frames_by_b_for_val(imgs, msks, boxes, valid, metas, epoch: int):
    B, T = imgs.shape[:2]
    prompt_frames_by_b = []

    for b in range(B):
        meta0 = metas[b] if (isinstance(metas, list) and len(metas) > b) else {}
        rng = _seeded_rng_for_val(epoch, meta0)

        seq_valid = valid[b].detach().cpu().numpy().astype(bool)
        seq_ne = (msks[b].sum(dim=(1, 2, 3)) > 0).detach().cpu().numpy().astype(bool)

        eligible = np.where(seq_valid & seq_ne)[0].astype(int)

        if eligible.size == 0:
            prompt_frames_by_b.append([])
            continue

        if T == 1:
            prompt_frames_by_b.append([int(eligible[0])])
            continue

        p = float(PROMPT_FRAME_PROB_3D)

        keep = rng.rand(eligible.shape[0]) < p
        chosen = eligible[keep]

        if chosen.size == 0:
            chosen = np.array([eligible[rng.randint(0, eligible.shape[0])]], dtype=int)

        chosen = np.unique(chosen)
        chosen.sort()
        prompt_frames_by_b.append([int(x) for x in chosen.tolist()])

    return prompt_frames_by_b

def _build_prompt_mode_by_b_train(B):
    return [_sample_prompt_mode_train() for _ in range(B)]

def _build_prompt_mode_by_b_val(metas, epoch):
    out = []
    for b in range(len(metas)):
        meta0 = metas[b] if metas is not None else {}
        out.append(_sample_prompt_mode_val(epoch, meta0))
    return out

def _forward_batch(model, predictor, imgs, msks, boxes, valid, metas=None, epoch=None, is_val=False):
    B, T = imgs.shape[:2]

    if T == 1:
        prompt_mode_by_b = _build_prompt_mode_by_b_val(metas, epoch) if is_val else _build_prompt_mode_by_b_train(B)
        logits_list = []
        for b in range(B):
            pmode = prompt_mode_by_b[b]
            logits_b = forward_oxford_prompt(
                model,
                imgs[b:b+1, 0],
                boxes_b4=boxes[b:b+1, 0],
                mask_prompts_b1hw=msks[b:b+1, 0],
                prompt_mode=pmode,
            )
            logits_list.append(logits_b)
        logits = torch.cat(logits_list, dim=0)
        return logits.unsqueeze(1)

    nonempty_bt = (msks.sum(dim=(2, 3, 4)) > 0)

    prompt_frames_by_b = None
    if is_val:
        assert metas is not None and epoch is not None
        prompt_frames_by_b = _build_prompt_frames_by_b_for_val(
            imgs=imgs, msks=msks, boxes=boxes, valid=valid, metas=metas, epoch=int(epoch)
        )

    prompt_mode_by_b = _build_prompt_mode_by_b_val(metas, epoch) if is_val else _build_prompt_mode_by_b_train(B)

    return forward_oxford_video_predictor_sequence(
        predictor=predictor,
        imgs_bt3hw_01=imgs,
        boxes_bt4=boxes,
        msks_bt1hw=msks,
        valid_bt=valid,
        nonempty_bt=nonempty_bt,
        obj_id=1,
        prompt_frames_by_b=prompt_frames_by_b,
        prompt_mode_by_b=prompt_mode_by_b,
    )

def _tb_log_vis(
    writer,
    epoch: int,
    kind: str,
    img_seq_t3hw,
    mask_seq_t1hw,
    logits_seq_thw,
    tag_base: str,
    gif_path: str,
    fps: int = 6,
    ping_pong: bool = True,
):
    frames = None

    try:
        out = save_vis_gif(
            img_seq_t3hw,
            mask_seq_t1hw,
            logits_seq_thw,
            gif_path,
            fps=fps,
            threshold=0.5,
            ping_pong=ping_pong,
        )
        if isinstance(out, (tuple, list)) and len(out) >= 2:
            frames = out[1]
        else:
            frames = out
    except Exception as e:
        writer.add_text("val/vis_error", tb_pre(f"[Epoch {epoch}] save_vis_gif(positional) failed ({kind}):\n{repr(e)}"), epoch)
        frames = None

    if frames is None:
        try:
            frames = make_vis_frames(
                img_seq_t3hw,
                mask_seq_t1hw,
                logits_seq_thw,
                threshold=0.5,
                ping_pong=ping_pong,
            )
            try:
                save_vis_gif(frames, gif_path, fps=fps)
            except TypeError:
                save_vis_gif(frames, gif_path, fps)
        except Exception as e:
            writer.add_text("val/vis_error", tb_pre(f"[Epoch {epoch}] make_vis_frames/save_vis_gif failed ({kind}):\n{repr(e)}"), epoch)
            return False

    try:
        writer.add_video(
            tag_base,
            frames_to_tb_video(frames),
            global_step=epoch,
            fps=fps,
        )
        return True
    except Exception as e:
        writer.add_text("val/vis_error", tb_pre(f"[Epoch {epoch}] add_video failed ({kind}):\n{repr(e)}"), epoch)
        return False

def train_epoch(model, predictor, train_loader, optimizer, scheduler, scaler, epoch, writer, global_step, LOG_EVERY_STEPS=25):
    model.train()

    losses = []
    dices  = []

    total_batches = len(train_loader)
    epoch_start_time = time.perf_counter()
    end_prev = time.perf_counter()

    pbar = tqdm(train_loader, desc=f"Train Epoch {epoch}", unit="batch", leave=True)

    for step, batch in enumerate(pbar, 1):
        data_wait = time.perf_counter() - end_prev
        if batch is None:
            end_prev = time.perf_counter()
            continue

        imgs, msks, boxes, valid, metas = batch
        imgs  = imgs.to(DEVICE, non_blocking=True)
        msks  = msks.to(DEVICE, non_blocking=True)
        boxes = boxes.to(DEVICE, non_blocking=True)
        valid = valid.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        t0 = time.perf_counter()

        with torch.cuda.amp.autocast(enabled=AMP_ENABLED, dtype=AMP_DTYPE):
            logits_bt1hw = _forward_batch(
                model=model,
                predictor=predictor,
                imgs=imgs,
                msks=msks,
                boxes=boxes,
                valid=valid,
                metas=None,
                epoch=None,
                is_val=False,
            )

            flat_logits  = logits_bt1hw.reshape(-1, 1, logits_bt1hw.shape[-2], logits_bt1hw.shape[-1])
            flat_msks    = msks.reshape(-1, 1, msks.shape[-2], msks.shape[-1])
            flat_valid   = valid.reshape(-1)

            flat_logits, flat_msks = _select_valid_frames(flat_logits, flat_msks, flat_valid)
            if flat_logits is None:
                end_prev = time.perf_counter()
                continue

            loss = bce_dice_loss(flat_logits, flat_msks)
            dice = dice_torch_from_logits(flat_logits, flat_msks)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        if torch.cuda.is_available():
            torch.cuda.synchronize()
        step_time = time.perf_counter() - t0

        losses.append(float(loss.detach().cpu().item()))
        dices.append(float(dice.detach().cpu().item()))

        pbar.set_postfix(
            loss=f"{losses[-1]:.4f}",
            dice=f"{dices[-1]:.4f}",
            data=f"{data_wait:.3f}s",
            step=f"{step_time:.3f}s",
        )

        if (step % LOG_EVERY_STEPS) == 0:
            writer.add_scalar("train/loss", float(np.mean(losses[-LOG_EVERY_STEPS:])), global_step)
            writer.add_scalar("train/dice@0.5", float(np.mean(dices[-LOG_EVERY_STEPS:])), global_step)
            writer.add_scalar("train/lr_group0", get_lr_group(optimizer, 0), global_step)
            writer.add_scalar("train/lr_group1", get_lr_group(optimizer, 1), global_step)
            writer.add_scalar("train/time_data_s", float(data_wait), global_step)
            writer.add_scalar("train/time_step_s", float(step_time), global_step)

        if (step % TEXT_PROGRESS_EVERY_STEPS) == 0 or (step == 1) or (step == total_batches):
            elapsed = time.perf_counter() - epoch_start_time
            pct_epoch = 100.0 * step / max(1, total_batches)
            text_blob = (
                f"epoch            : {epoch + 1}/{EPOCHS}\n"
                f"step             : {step}/{total_batches}\n"
                f"epoch_progress   : {pct_epoch:.2f}%\n"
                f"global_step      : {global_step}\n"
                f"latest_loss      : {losses[-1]:.6f}\n"
                f"latest_dice@0.5  : {dices[-1]:.6f}\n"
                f"lr_group0        : {get_lr_group(optimizer, 0):.8f}\n"
                f"lr_group1        : {get_lr_group(optimizer, 1):.8f}\n"
                f"elapsed_sec      : {elapsed:.2f}\n"
                f"avg_loss_so_far  : {float(np.mean(losses)):.6f}\n"
                f"avg_dice_so_far  : {float(np.mean(dices)):.6f}"
            )
            writer.add_text("train/progress", tb_pre(text_blob), global_step)

        global_step += 1
        end_prev = time.perf_counter()

    avg_loss = float(np.mean(losses)) if losses else 0.0
    avg_dice = float(np.mean(dices))  if dices  else 0.0

    writer.add_scalar("train_epoch/loss", avg_loss, epoch)
    writer.add_scalar("train_epoch/dice@0.5", avg_dice, epoch)
    writer.add_text(
        "train/epoch_summary",
        tb_pre(
            f"epoch            : {epoch + 1}/{EPOCHS}\n"
            f"batches          : {total_batches}\n"
            f"avg_loss         : {avg_loss:.6f}\n"
            f"avg_dice@0.5     : {avg_dice:.6f}\n"
            f"lr_group0_end    : {get_lr_group(optimizer, 0):.8f}\n"
            f"lr_group1_end    : {get_lr_group(optimizer, 1):.8f}"
        ),
        epoch,
    )
    return avg_loss, avg_dice, global_step

@torch.no_grad()
def validate_epoch(
    model,
    predictor,
    loader,
    epoch,
    writer,
    max_batches=None,
    modalities_map=None,
):
    model.eval()

    try:
        tasks_map = json.load(open(config.dripp.tasks_file, "r"))
        tasks_map = {str(k): v for k, v in tasks_map.items()}
    except Exception:
        tasks_map = {}

    task_agg = TaskAggregator(tasks_map=tasks_map)
    mod_agg  = ModalityAggregator(modalities_map=modalities_map)

    vol_keys = set(getattr(task_agg, "_vol_keys", set()))

    totals = {}
    counts = {}
    n_slices_total = 0
    n_vols_total   = 0

    logged_2d = False
    logged_3d = False
    vis_tasks_done = set()

    E_total = int(globals().get("EPOCHS", epoch + 1))
    is_final_epoch = (epoch == (E_total - 1))
    do_viz_this_ep = ((epoch % int(globals().get("VIS_EVERY_EPOCHS", 5))) == 0) or is_final_epoch

    pbar = tqdm(loader, desc=f"Val Epoch {epoch}", unit="batch", leave=False)
    for bi, batch in enumerate(pbar, 1):
        if batch is None:
            continue

        imgs, msks, boxes, valid, metas = batch
        imgs  = imgs.to(DEVICE, non_blocking=True)
        msks  = msks.to(DEVICE, non_blocking=True)
        boxes = boxes.to(DEVICE, non_blocking=True)
        valid = valid.to(DEVICE, non_blocking=True)

        B, T = imgs.shape[:2]

        with torch.cuda.amp.autocast(enabled=AMP_ENABLED, dtype=AMP_DTYPE):
            logits_bt1hw = _forward_batch(
                model=model,
                predictor=predictor,
                imgs=imgs,
                msks=msks,
                boxes=boxes,
                valid=valid,
                metas=metas,
                epoch=int(epoch),
                is_val=True,
            )

        probs_bt = torch.sigmoid(logits_bt1hw).float()
        gt_bt    = (msks > 0.5)

        for b in range(B):
            v_idx = torch.nonzero(valid[b], as_tuple=False).squeeze(1)
            if v_idx.numel() == 0:
                continue

            probs_v = probs_bt[b].index_select(0, v_idx)[:, 0]
            gt_v    = gt_bt[b].index_select(0, v_idx)[:, 0]
            Tv = int(probs_v.shape[0])

            if Tv == 1:
                m = compute_metrics_2d(
                    probs_flat=probs_v,
                    masks_flat=gt_v,
                    idx_list=[0],
                    n_valid=1,
                )
                dim = 2
            else:
                m = compute_metrics_3d(
                    probs=probs_v,
                    gts=gt_v,
                    idx_list=list(range(Tv)),
                    n_valid=Tv,
                    batch={},
                )
                dim = 3

            meta0 = metas[b] if (isinstance(metas, list) and len(metas) > b) else {}

            t_id  = meta0.get("task_id", None)
            t_lab = meta0.get("task_label", None)

            if isinstance(t_id, (list, tuple)):
                t_id = t_id[0] if len(t_id) > 0 else None
            if isinstance(t_lab, (list, tuple)):
                t_lab = t_lab[0] if len(t_lab) > 0 else None

            if t_id is not None:
                t_id = str(t_id)

            task_label = task_agg.label_for(t_id if t_id is not None else t_lab)

            mod_label = str(
                meta0.get("modality", None)
                or meta0.get("subdataset_modality", None)
                or "unknown"
            )

            task_agg.update(task_label, m, slice_weight=Tv, vol_weight=1, dim=dim, n_items=1)
            mod_agg.update(mod_label,  m, slice_weight=Tv, vol_weight=1, dim=dim, n_items=1)

            for k, v in m.items():
                if not isinstance(v, (int, float, np.floating)):
                    continue
                if v != v:
                    continue
                w = 1.0 if (k in vol_keys) else float(Tv)
                totals[k] = totals.get(k, 0.0) + float(v) * w
                counts[k] = counts.get(k, 0.0) + w

            n_slices_total += Tv
            n_vols_total   += 1

            postfix = {"slices": n_slices_total, "vols": n_vols_total}
            if "dice@0.5" in totals and counts.get("dice@0.5", 0) > 0:
                postfix["dice@0.5"] = f"{(totals['dice@0.5']/counts['dice@0.5']):.4f}"
            if "iou@0.5" in totals and counts.get("iou@0.5", 0) > 0:
                postfix["iou@0.5"] = f"{(totals['iou@0.5']/counts['iou@0.5']):.4f}"
            pbar.set_postfix(**postfix)

            visualize_this_item = False
            if do_viz_this_ep:
                if not is_final_epoch:
                    visualize_this_item = ((dim == 2) and (not logged_2d)) or ((dim == 3) and (not logged_3d))
                else:
                    visualize_this_item = (task_label not in vis_tasks_done)

            if visualize_this_item:
                img_seq    = imgs[b].index_select(0, v_idx).detach().cpu()
                mask_seq   = msks[b].index_select(0, v_idx).detach().cpu()
                logits_seq = logits_bt1hw[b].index_select(0, v_idx)[:, 0].detach().cpu()

                safe_task = _safe_tag(task_label)
                kind      = "2d" if dim == 2 else "3d"

                if is_final_epoch:
                    gif_name = f"val{kind}_task-{safe_task}_ep{epoch:04d}.gif"
                    tag      = f"val{kind}/vis/{safe_task}"
                    vis_tasks_done.add(task_label)
                else:
                    gif_name = f"val{kind}_ep{epoch:04d}.gif"
                    tag      = f"val{kind}/vis"
                    if dim == 2:
                        logged_2d = True
                    else:
                        logged_3d = True

                gif_path = os.path.join(VIS_DIR, gif_name)

                _tb_log_vis(
                    writer=writer,
                    epoch=epoch,
                    kind=kind,
                    img_seq_t3hw=img_seq,
                    mask_seq_t1hw=mask_seq,
                    logits_seq_thw=logits_seq,
                    tag_base=tag,
                    gif_path=gif_path,
                    fps=VIS_FPS,
                    ping_pong=VIS_PING_PONG,
                )

        if (max_batches is not None) and (bi >= max_batches):
            break

    results = {}
    for k in sorted(totals.keys()):
        c = float(counts.get(k, 0.0))
        if c > 0:
            results[k] = float(totals[k] / c)

    results["n_slices"]  = int(n_slices_total)
    results["n_volumes"] = int(n_vols_total)

    for k, v in results.items():
        if isinstance(v, (int, float, np.floating)) and (v == v):
            writer.add_scalar(f"val/{k}", float(v), epoch)
    writer.add_text("val/summary", tb_pre(json.dumps(results, indent=2)), epoch)

    try:
        task_agg.log_to_tensorboard(writer, epoch, split="val")
        writer.add_text("val/per_task_table", tb_pre(task_agg.format_text_table()), epoch)
    except Exception as e:
        writer.add_text("val/task_agg_error", tb_pre(repr(e)), epoch)

    try:
        mod_agg.log_to_tensorboard(writer, epoch, split="val")
        writer.add_text("val/per_modality_table", tb_pre(mod_agg.format_text_table()), epoch)
    except Exception as e:
        writer.add_text("val/mod_agg_error", tb_pre(repr(e)), epoch)

    writer.flush()
    model.train()
    return results

global_step = 0
last_val_metrics = None
best_val_score = -float("inf")
best_ckpt_path = os.path.join(CKPT_OUT, "best_val.pth")
MAX_VAL_BATCHES = None

for epoch in range(EPOCHS):
    train_loader = make_train_loader(epoch)

    avg_loss, avg_dice, global_step = train_epoch(
        model=model,
        predictor=predictor,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
        epoch=epoch,
        writer=writer,
        global_step=global_step,
        LOG_EVERY_STEPS=LOG_EVERY_STEPS,
    )
    print(f"[Epoch {epoch}] train: loss={avg_loss:.4f} dice@0.5={avg_dice:.4f}")

    do_val = (epoch == 0) or (((epoch + 1) % VAL_EVERY_EPOCHS) == 0) or (epoch == (EPOCHS - 1))
    val_metrics = None
    if do_val:
        val_metrics = validate_epoch(
            model=model,
            predictor=predictor,
            loader=val_loader,
            epoch=epoch,
            writer=writer,
            max_batches=MAX_VAL_BATCHES,
            modalities_map=None,
        )
        last_val_metrics = val_metrics
        print(f"[Epoch {epoch}] val: {val_metrics}")

        score = None
        for k in ["iou_best", "dice_best", "iou@0.5", "dice@0.5"]:
            if k in val_metrics and isinstance(val_metrics[k], (int, float)) and val_metrics[k] == val_metrics[k]:
                score = float(val_metrics[k])
                break

        if score is not None and score > best_val_score:
            best_val_score = score
            torch.save(
                {
                    "epoch": epoch,
                    "model": model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "global_step": global_step,
                    "val_metrics": val_metrics,
                    "best_val_score": best_val_score,
                },
                best_ckpt_path,
            )
            print(f"[Epoch {epoch}] new best checkpoint: {best_ckpt_path} (score={best_val_score:.6f})")
    else:
        print(f"[Epoch {epoch}] val: skipped (every {VAL_EVERY_EPOCHS} epochs + first + final)")

    ckpt_path = os.path.join(CKPT_OUT, f"epoch_{epoch:03d}.pth")
    torch.save(
        {
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scheduler": scheduler.state_dict(),
            "global_step": global_step,
            "val_metrics": val_metrics,
        },
        ckpt_path,
    )
    print(f"[Epoch {epoch}] saved: {ckpt_path}")

writer.flush()
writer.close()
print("Done.")

Train Epoch 0:   0%|          | 0/152825 [00:00<?, ?batch/s]

/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:182: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(


In [1]:
# =========================
# EVAL CELL (REPLACEMENT)
# - prefers best_val.pth if present
# - runs 3 eval modes:
#     1) box-only
#     2) mask-only
#     3) mixed 50/50
# =========================
import os, glob, json
import torch
from torch.utils.tensorboard import SummaryWriter

assert "CKPT_OUT" in globals(), "CKPT_OUT not found."
assert "val_loader" in globals(), "val_loader not found."
assert "model" in globals(), "model not found."
assert "predictor" in globals(), "predictor not found."
assert "validate_epoch" in globals(), "validate_epoch not found."
assert "_sample_prompt_mode_val" in globals(), "_sample_prompt_mode_val not found."
assert "VAL_PROMPT_TYPE_PROBS" in globals(), "VAL_PROMPT_TYPE_PROBS not found."

def _pick_eval_ckpt(ckpt_dir):
    best_path = os.path.join(ckpt_dir, "best_val.pth")
    if os.path.isfile(best_path):
        return best_path
    cands = sorted(glob.glob(os.path.join(ckpt_dir, "epoch_*.pth")))
    if not cands:
        raise FileNotFoundError(f"No checkpoint found under: {ckpt_dir}")
    cands.sort(key=lambda p: os.path.getmtime(p))
    return cands[-1]

def _load_ckpt_into_model(model, ckpt_path):
    obj = torch.load(ckpt_path, map_location="cpu")
    sd = obj.get("model", obj)
    if isinstance(sd, dict) and any(k.startswith("module.") for k in sd.keys()):
        sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
    missing, unexpected = model.load_state_dict(sd, strict=False)
    return obj, missing, unexpected

def _print_metrics_block(title, metrics, priority):
    print(f"\n[{title}]")
    for k in priority:
        if k in metrics:
            v = metrics[k]
            if isinstance(v, (int, float)):
                print(f"  {k:22s} {v:.6f}")
            else:
                print(f"  {k:22s} {v}")

    rest = sorted([k for k in metrics.keys() if k not in set(priority)])
    if rest:
        print("\n  additional metrics:")
        for k in rest:
            v = metrics[k]
            if isinstance(v, (int, float)):
                print(f"  {k:22s} {v:.6f}")
            else:
                print(f"  {k:22s} {v}")

def _run_eval_mode(mode_name, val_prompt_probs, epoch_for_eval):
    global VAL_PROMPT_TYPE_PROBS

    old_val_probs = dict(VAL_PROMPT_TYPE_PROBS)

    try:
        VAL_PROMPT_TYPE_PROBS = dict(val_prompt_probs)

        eval_tb_dir_mode = os.path.join(EVAL_TB_DIR, mode_name)
        os.makedirs(eval_tb_dir_mode, exist_ok=True)
        eval_writer = SummaryWriter(log_dir=eval_tb_dir_mode, flush_secs=5, max_queue=10)

        model.eval()
        with torch.no_grad():
            metrics = validate_epoch(
                model=model,
                predictor=predictor,
                loader=val_loader,
                epoch=epoch_for_eval,
                writer=eval_writer,
                max_batches=None,
                modalities_map=None,
            )

        eval_writer.flush()
        eval_writer.close()
        model.train()
        return metrics

    finally:
        VAL_PROMPT_TYPE_PROBS = old_val_probs

ckpt_path = _pick_eval_ckpt(CKPT_OUT)
ckpt_obj, missing, unexpected = _load_ckpt_into_model(model, ckpt_path)

epoch_in_ckpt = ckpt_obj.get("epoch", None)
print(f"[EVAL] Loaded checkpoint: {ckpt_path}")
print(f"[EVAL] epoch in ckpt: {epoch_in_ckpt}")
print(f"[EVAL] load_state_dict(strict=False): missing={len(missing)} unexpected={len(unexpected)}")

if epoch_in_ckpt is not None:
    epoch_for_eval = int(epoch_in_ckpt)
else:
    epoch_for_eval = int(globals().get("EPOCHS", 1)) - 1
epoch_for_eval = max(0, epoch_for_eval)

EVAL_TB_DIR = os.path.join(OUT_ROOT, "tb_eval_multi")
os.makedirs(EVAL_TB_DIR, exist_ok=True)

priority = [
    "dice@0.5", "iou@0.5",
    "dice_best", "iou_best",
    "dice@avg_thr", "iou@avg_thr",
    "roc_auc", "pr_auc",
    "ece",
    "iou3d@0.5", "dice3d@0.5",
    "iou3d@best", "dice3d@best",
    "best_thr", "best_thr3d",
    "n_slices", "n_volumes",
]

eval_modes = {
    "box_only":  {"box": 1.0, "mask": 0.0},
    "mask_only": {"box": 0.0, "mask": 1.0},
    "mixed_50_50": {"box": 0.5, "mask": 0.5},
}

all_eval_results = {}

for mode_name, prompt_probs in eval_modes.items():
    print(f"\n[EVAL] Running mode='{mode_name}' with VAL_PROMPT_TYPE_PROBS={prompt_probs}")
    metrics = _run_eval_mode(
        mode_name=mode_name,
        val_prompt_probs=prompt_probs,
        epoch_for_eval=epoch_for_eval,
    )
    all_eval_results[mode_name] = metrics

for mode_name, metrics in all_eval_results.items():
    _print_metrics_block(
        title=f"OXFORD-REFINE EVAL :: {mode_name}",
        metrics=metrics,
        priority=priority,
    )

print(f"\n[EVAL] TensorBoard eval logs written to: {EVAL_TB_DIR}")

AssertionError: CKPT_OUT not found.

In [ ]:
# %%
# =========================
# STANDALONE BALANCED SUBSAMPLED VALIDATION CELL
# FULL TASK COVERAGE + FULL DETAILED METRICS + ONE VIS PER TASK
# - no prior notebook cells required
# - samples validation like the older balanced loader cell
# - box-only validation
# - includes overall / per-task / per-modality metrics
# =========================
import os, sys, re, math, json, glob, time, html, pickle, types, warnings
import importlib.util, importlib.machinery
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

import SimpleITK as sitk
from PIL import Image

from tqdm.auto import tqdm

# -------------------------------------------------
# EDIT THESE IF NEEDED
# -------------------------------------------------
LOADERS_PKL = "/data/loaders32.pkl"

RUN_DIR = "/data/jrbonill/oxford_refine_runs"
SPECIFIC_RUN = None
CHECKPOINT_PATH = None
CHECKPOINT_PREFERENCE = "best"   # "best" or "latest_epoch"

REPO_ROOT_CANDIDATES = [
    "/data/jrbonill/RWKV-MedSAM2",
    "/data/jrbonill/rwkv_medsam2",
    "/RWKV-MedSAM2",
]

OXFORD_ENV_BASE = "/data/jrbonill/oxford_medsam2_env"
SAM_CKPT = f"{OXFORD_ENV_BASE}/Medical-SAM2/checkpoints/sam2_hiera_tiny.pt"

TARGET_HW = 512
SEED = 42

# Requested balanced sub-val config
SUBVAL_NUM_ITEMS = 150
SUBVAL_MIN_PER_BUCKET = 2
SUBVAL_BATCH_SIZE = 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

# -------------------------------------------------
# Determinism
# -------------------------------------------------
def set_determinism(seed: int):
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

set_determinism(SEED)

warnings.filterwarnings(
    "ignore",
    message=re.escape("Deterministic behavior was enabled") + ".*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=re.escape("Memory Efficient attention defaults to a non-deterministic algorithm") + ".*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=re.escape("Flash Attention defaults to a non-deterministic algorithm") + ".*",
    category=UserWarning,
)
warnings.filterwarnings(
    "ignore",
    message=re.escape("upsample_bicubic2d_backward_out_cuda does not have a deterministic implementation") + ".*",
    category=UserWarning,
)

# -------------------------------------------------
# Repo / safe package loading
# -------------------------------------------------
def _find_repo_root():
    for root in REPO_ROOT_CANDIDATES:
        if os.path.isdir(root) and os.path.isfile(os.path.join(root, "rwkv_medsam2", "dataset.py")):
            return os.path.abspath(root)
    return None

REPO_ROOT = _find_repo_root()
assert REPO_ROOT is not None, f"Could not locate RWKV-MedSAM2 repo from: {REPO_ROOT_CANDIDATES}"
PKG_ROOT = os.path.join(REPO_ROOT, "rwkv_medsam2")
assert os.path.isdir(PKG_ROOT), f"Missing package dir: {PKG_ROOT}"

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

def _load_module_from_path(module_name, path):
    loader = importlib.machinery.SourceFileLoader(module_name, path)
    spec = importlib.util.spec_from_loader(loader.name, loader)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    loader.exec_module(mod)
    return mod

def _install_safe_rwkv_package():
    for name in list(sys.modules.keys()):
        if name == "rwkv_medsam2" or name.startswith("rwkv_medsam2."):
            del sys.modules[name]

    pkg = types.ModuleType("rwkv_medsam2")
    pkg.__path__ = [PKG_ROOT]
    pkg.__file__ = os.path.join(PKG_ROOT, "__init__.py")
    sys.modules["rwkv_medsam2"] = pkg

    dataset_py = os.path.join(PKG_ROOT, "dataset.py")
    dataset_mod = _load_module_from_path("rwkv_medsam2.dataset", dataset_py)
    setattr(pkg, "dataset", dataset_mod)

_install_safe_rwkv_package()
from rwkv_medsam2.dataset import BalancedTaskBatchSampler

# -------------------------------------------------
# vis.py helpers
# -------------------------------------------------
VIS_PY_CANDIDATES = [
    os.path.join(os.getcwd(), "vis.py"),
    "./vis.py",
    "/data/jrbonill/oxford_medsam2_env/Medical-SAM2/vis.py",
    "/data/jrbonill/oxford_medsam2_env/site/vis.py",
    "/data/jrbonill/oxford_medsam2_env/vis.py",
    "/data/jrbonill/vis.py",
    "/RWKV-MedSAM2/rwkv_medsam2/utils/vis.py",
]

def _find_vis_py():
    for p in VIS_PY_CANDIDATES:
        if p and os.path.isfile(p):
            return os.path.abspath(p)
    return None

def load_vis_helpers():
    vis_path = _find_vis_py()
    if vis_path is None:
        raise RuntimeError("Could not locate vis.py")

    name = "vis_standalone_for_subval_full"
    loader = importlib.machinery.SourceFileLoader(name, vis_path)
    spec = importlib.util.spec_from_loader(loader.name, loader)
    mod = importlib.util.module_from_spec(spec)
    loader.exec_module(mod)

    needed = ["make_vis_frames", "save_vis_gif", "frames_to_tb_video"]
    missing = [k for k in needed if not hasattr(mod, k)]
    if missing:
        raise RuntimeError(f"vis.py at {vis_path} missing: {missing}")

    return mod.make_vis_frames, mod.save_vis_gif, mod.frames_to_tb_video

make_vis_frames, save_vis_gif, frames_to_tb_video = load_vis_helpers()

# -------------------------------------------------
# Metric helpers from current func_metrics.py
# -------------------------------------------------
def _find_rwkv_func_metrics_py():
    for root in REPO_ROOT_CANDIDATES:
        p = os.path.join(root, "rwkv_medsam2", "functions", "func_metrics.py")
        if os.path.isfile(p):
            return p
    return None

def load_metrics_utils():
    wanted = [
        "TaskAggregator", "ModalityAggregator",
        "dice_iou", "ece", "try_auc", "fg_bin", "sigmoid_np",
    ]
    try:
        from rwkv_medsam2.functions.func_metrics import (
            TaskAggregator, ModalityAggregator,
            dice_iou, ece, try_auc, fg_bin, sigmoid_np,
        )
        return TaskAggregator, ModalityAggregator, dice_iou, ece, try_auc, fg_bin, sigmoid_np
    except Exception as e:
        fm_path = _find_rwkv_func_metrics_py()
        if fm_path is None:
            raise RuntimeError("Could not find func_metrics.py") from e

        name = "rwkv_medsam2_func_metrics_standalone_for_subval"
        loader = importlib.machinery.SourceFileLoader(name, fm_path)
        spec = importlib.util.spec_from_loader(loader.name, loader)
        mod = importlib.util.module_from_spec(spec)
        loader.exec_module(mod)

        missing = [k for k in wanted if not hasattr(mod, k)]
        if missing:
            raise RuntimeError(f"Missing {missing} in {fm_path}")

        return (
            mod.TaskAggregator,
            mod.ModalityAggregator,
            mod.dice_iou,
            mod.ece,
            mod.try_auc,
            mod.fg_bin,
            mod.sigmoid_np,
        )

(
    TaskAggregator,
    ModalityAggregator,
    dice_iou,
    ece,
    try_auc,
    fg_bin,
    sigmoid_np,
) = load_metrics_utils()

_METRIC_THR_PROBS = np.linspace(0.05, 0.95, 19, dtype=np.float64)
_METRIC_THR_LOGITS = np.log(_METRIC_THR_PROBS / (1.0 - _METRIC_THR_PROBS)).astype(np.float64)

def _as_numpy_bool_mask(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    return (x > 0.5).astype(np.uint8)

def _as_numpy_probs(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    x = np.asarray(x)
    return x.astype(np.float32)

def _slice_by_idx(arr, idx_list=None, n_valid=None):
    arr = np.asarray(arr)
    if idx_list is not None:
        idx = np.asarray(list(idx_list), dtype=np.int64)
        if idx.size > 0:
            arr = arr[idx]
    elif n_valid is not None:
        arr = arr[:int(n_valid)]
    return arr

def _metrics_from_logits_and_gt(logits_np, gt_np):
    logits_np = np.asarray(logits_np, dtype=np.float32)
    gt_np = np.asarray(gt_np, dtype=np.uint8)

    if logits_np.ndim == 2:
        logits_np = logits_np[None, ...]
    if gt_np.ndim == 2:
        gt_np = gt_np[None, ...]

    probs_np = sigmoid_np(logits_np)
    pred05 = (logits_np > 0.0).astype(np.uint8)

    dices = []
    ious = []
    fg_fracs = []
    for kslice in range(gt_np.shape[0]):
        d, i = dice_iou(torch.from_numpy(pred05[kslice]), torch.from_numpy(gt_np[kslice]))
        dices.append(float(d))
        ious.append(float(i))
        fg_fracs.append(float(gt_np[kslice].mean()))
    dice05 = float(np.mean(dices)) if dices else float("nan")
    iou05 = float(np.mean(ious)) if ious else float("nan")

    avg_d_list = []
    avg_i_list = []
    best_d = -1.0
    best_i = -1.0
    best_thr = float(_METRIC_THR_LOGITS[0])

    for thr in _METRIC_THR_LOGITS:
        pred = (logits_np > float(thr)).astype(np.uint8)
        d_thr = []
        i_thr = []
        for kslice in range(gt_np.shape[0]):
            d, i = dice_iou(torch.from_numpy(pred[kslice]), torch.from_numpy(gt_np[kslice]))
            d_thr.append(float(d))
            i_thr.append(float(i))
        d_m = float(np.mean(d_thr)) if d_thr else float("nan")
        i_m = float(np.mean(i_thr)) if i_thr else float("nan")
        avg_d_list.append(d_m)
        avg_i_list.append(i_m)
        if d_m > best_d:
            best_d = d_m
            best_i = i_m
            best_thr = float(thr)

    dice_avg = float(np.mean(avg_d_list)) if avg_d_list else float("nan")
    iou_avg = float(np.mean(avg_i_list)) if avg_i_list else float("nan")

    flat_probs = probs_np.reshape(-1).astype(np.float64)
    flat_gt = gt_np.reshape(-1).astype(np.uint8)
    ece_val = float(ece(flat_probs, flat_gt, n_bins=15)) if flat_probs.size > 0 else float("nan")
    roc_auc, pr_auc = try_auc(flat_gt, flat_probs) if flat_probs.size > 0 else (float("nan"), float("nan"))
    roc_auc = float(roc_auc) if roc_auc == roc_auc else float("nan")
    pr_auc = float(pr_auc) if pr_auc == pr_auc else float("nan")

    out = {
        "dice@0.5": dice05,
        "iou@0.5": iou05,
        "dice@avg_thr": dice_avg,
        "iou@avg_thr": iou_avg,
        "dice_best": float(best_d),
        "iou_best": float(best_i),
        "best_thr": float(best_thr),
        "ece": ece_val,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "n": int(gt_np.shape[0]),
    }

    for frac, d, i in zip(fg_fracs, dices, ious):
        key = fg_bin(frac)
        out[f"{key}/dice@0.5"] = float(d)
        out[f"{key}/iou@0.5"] = float(i)

    return out, probs_np, gt_np

def compute_metrics_2d(probs_flat, masks_flat, idx_list=None, n_valid=None, **kwargs):
    probs_np = _slice_by_idx(_as_numpy_probs(probs_flat), idx_list=idx_list, n_valid=n_valid)
    gt_np = _slice_by_idx(_as_numpy_bool_mask(masks_flat), idx_list=idx_list, n_valid=n_valid)

    if probs_np.ndim == 2:
        probs_np = probs_np[None, ...]
    if gt_np.ndim == 2:
        gt_np = gt_np[None, ...]

    probs_np = np.clip(probs_np.astype(np.float32), 1e-6, 1.0 - 1e-6)
    logits_np = np.log(probs_np / (1.0 - probs_np)).astype(np.float32)

    out, _, gt_np = _metrics_from_logits_and_gt(logits_np, gt_np)
    out["n"] = int(gt_np.shape[0])
    return out

def compute_metrics_3d(probs, gts, idx_list=None, n_valid=None, batch=None, **kwargs):
    probs_np = _slice_by_idx(_as_numpy_probs(probs), idx_list=idx_list, n_valid=n_valid)
    gt_np = _slice_by_idx(_as_numpy_bool_mask(gts), idx_list=idx_list, n_valid=n_valid)

    if probs_np.ndim == 2:
        probs_np = probs_np[None, ...]
    if gt_np.ndim == 2:
        gt_np = gt_np[None, ...]

    probs_np = np.clip(probs_np.astype(np.float32), 1e-6, 1.0 - 1e-6)
    logits_np = np.log(probs_np / (1.0 - probs_np)).astype(np.float32)

    out, _, gt_np = _metrics_from_logits_and_gt(logits_np, gt_np)

    out["dice3d@0.5"] = float(out["dice@0.5"])
    out["iou3d@0.5"] = float(out["iou@0.5"])
    out["dice3d@best"] = float(out["dice_best"])
    out["iou3d@best"] = float(out["iou_best"])

    vol_mask = (gt_np.sum(axis=(1, 2)) > 0)
    if np.any(vol_mask):
        vL = logits_np[vol_mask]
        vG = gt_np[vol_mask]
        best_dv = -1.0
        best_iv = -1.0
        best_tv = float(_METRIC_THR_LOGITS[0])
        for thr in _METRIC_THR_LOGITS:
            predv = (vL > float(thr)).astype(np.uint8)
            d, i = dice_iou(torch.from_numpy(predv), torch.from_numpy(vG))
            d = float(d)
            i = float(i)
            if d > best_dv:
                best_dv = d
                best_iv = i
                best_tv = float(thr)
        out["dice3d@best_vol"] = float(best_dv)
        out["iou3d@best_vol"] = float(best_iv)
        out["best_thr3d"] = float(best_tv)
    else:
        out["dice3d@best_vol"] = float("nan")
        out["iou3d@best_vol"] = float("nan")
        out["best_thr3d"] = float("nan")

    out["n"] = int(gt_np.shape[0])
    out["n_vol"] = 1
    return out

# -------------------------------------------------
# Load saved loaders32.pkl safely
# -------------------------------------------------
def _looks_like_dataloader(x):
    return hasattr(x, "dataset") and hasattr(x, "batch_sampler") and hasattr(x, "__iter__") and hasattr(x, "__len__")

def _extract_loaders_from_dict(obj):
    preferred_key_sets = [
        ("train_loader", "val_loader", "test_loader"),
        ("train", "val", "test"),
        ("train_dataloader", "val_dataloader", "test_dataloader"),
    ]
    for ks in preferred_key_sets:
        if all(k in obj for k in ks):
            return obj[ks[0]], obj[ks[1]], obj[ks[2]]

    loader_like = {k: v for k, v in obj.items() if _looks_like_dataloader(v)}
    if len(loader_like) >= 3:
        def _pick_by_name(name_candidates):
            for k, v in loader_like.items():
                kl = str(k).lower()
                if any(tok in kl for tok in name_candidates):
                    return v
            return None
        train_loader = _pick_by_name(["train"])
        val_loader   = _pick_by_name(["val", "valid"])
        test_loader  = _pick_by_name(["test"])
        if train_loader is not None and val_loader is not None and test_loader is not None:
            return train_loader, val_loader, test_loader
        vals = list(loader_like.values())
        return vals[0], vals[1], vals[2]

    for _, v in obj.items():
        if isinstance(v, dict):
            nested = {kk: vv for kk, vv in v.items() if _looks_like_dataloader(vv)}
            if len(nested) >= 3:
                def _pick_nested(name_candidates):
                    for kk, vv in nested.items():
                        kkl = str(kk).lower()
                        if any(tok in kkl for tok in name_candidates):
                            return vv
                    return None
                train_loader = _pick_nested(["train"])
                val_loader   = _pick_nested(["val", "valid"])
                test_loader  = _pick_nested(["test"])
                if train_loader is not None and val_loader is not None and test_loader is not None:
                    return train_loader, val_loader, test_loader
                vals = list(nested.values())
                return vals[0], vals[1], vals[2]
    return None, None, None

def _load_saved_loaders(pkl_path):
    with open(pkl_path, "rb") as f:
        obj = pickle.load(f)
    if isinstance(obj, (tuple, list)) and len(obj) == 3:
        return obj[0], obj[1], obj[2]
    if isinstance(obj, dict):
        a, b, c = _extract_loaders_from_dict(obj)
        if a is not None:
            return a, b, c
    raise RuntimeError(f"Could not locate train/val/test loaders inside {pkl_path}")

train_loader_base, val_loader_base, test_loader_base = _load_saved_loaders(LOADERS_PKL)
val_sequences = list(val_loader_base.dataset.sequences)
print("Loaded val sequences:", len(val_sequences))

# -------------------------------------------------
# Helpers for Oxford-style adapter dataset
# -------------------------------------------------
def _ext(path):
    p = str(path).lower()
    if p.endswith(".nii.gz"):
        return ".nii.gz"
    return os.path.splitext(p)[1].lower()

IMG_EXT_2D = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}

def bbox_from_mask(mask_hw, pad=5):
    ys, xs = np.where(mask_hw > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    x0, x1 = xs.min(), xs.max()
    y0, y1 = ys.min(), ys.max()
    x0 = max(0, x0 - pad)
    y0 = max(0, y0 - pad)
    x1 = min(mask_hw.shape[1] - 1, x1 + pad)
    y1 = min(mask_hw.shape[0] - 1, y1 + pad)
    return [float(x0), float(y0), float(x1), float(y1)]

def resize_u8_gray(u8_hw, out_hw=512):
    t = torch.from_numpy(u8_hw).unsqueeze(0).unsqueeze(0).float()
    t = F.interpolate(t, size=(out_hw, out_hw), mode="bilinear", align_corners=False)
    return t.squeeze(0).squeeze(0).clamp(0, 255).byte().cpu().numpy()

def resize_mask_u8(mask_hw, out_hw=512):
    t = torch.from_numpy(mask_hw.astype(np.float32)).unsqueeze(0).unsqueeze(0)
    t = F.interpolate(t, size=(out_hw, out_hw), mode="nearest")
    return (t.squeeze(0).squeeze(0).cpu().numpy() > 0.5).astype(np.uint8)

def load_2d_gray_u8(path):
    ext = _ext(path)
    if ext in IMG_EXT_2D:
        im = Image.open(path).convert("L")
        return np.array(im, dtype=np.uint8)
    elif ext in [".nii", ".nii.gz"]:
        arr = sitk.GetArrayFromImage(sitk.ReadImage(path))
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(np.float32)
        lo, hi = np.percentile(arr, 0.5), np.percentile(arr, 99.5)
        if hi <= lo:
            hi = lo + 1.0
        arr = np.clip(arr, lo, hi)
        arr = (255.0 * (arr - lo) / (hi - lo)).clip(0, 255).astype(np.uint8)
        return arr
    elif ext == ".npy":
        arr = np.load(path)
        if arr.ndim == 3:
            arr = arr[0]
        arr = arr.astype(np.float32)
        lo, hi = np.percentile(arr, 0.5), np.percentile(arr, 99.5)
        if hi <= lo:
            hi = lo + 1.0
        arr = np.clip(arr, lo, hi)
        arr = (255.0 * (arr - lo) / (hi - lo)).clip(0, 255).astype(np.uint8)
        return arr
    raise ValueError(f"Unsupported image ext: {path}")

def load_2d_mask01(path):
    ext = _ext(path)
    if ext in IMG_EXT_2D:
        m = Image.open(path).convert("L")
        return (np.array(m) > 0).astype(np.uint8)
    elif ext in [".nii", ".nii.gz"]:
        arr = sitk.GetArrayFromImage(sitk.ReadImage(path))
        if arr.ndim == 3:
            arr = arr[0]
        return (arr > 0).astype(np.uint8)
    elif ext == ".npy":
        arr = np.load(path)
        if arr.ndim == 3:
            arr = arr[0]
        return (arr > 0).astype(np.uint8)
    raise ValueError(f"Unsupported mask ext: {path}")

def _extract_frame_pairs(seqrec):
    seq = seqrec.get("sequence", None)
    if isinstance(seq, list) and len(seq) > 0:
        out = []
        for item in seq:
            if isinstance(item, (tuple, list)) and len(item) >= 2:
                out.append((str(item[0]), str(item[1])))
            elif isinstance(item, dict):
                img = item.get("image", None)
                msk = item.get("mask", None)
                if img is not None and msk is not None:
                    out.append((str(img), str(msk)))
        if len(out) > 0:
            return out
    if seqrec.get("image") is not None and seqrec.get("mask") is not None:
        return [(str(seqrec["image"]), str(seqrec["mask"]))]
    return []

SEQ_LEN_3D = 8

class OxfordFromTrainingSequences(Dataset):
    def __init__(self, sequences, target_hw=512, pad=5, seq_len_3d=8, is_val=True, seed=42):
        self.sequences = sequences
        self.target_hw = int(target_hw)
        self.pad = int(pad)
        self.seq_len_3d = int(seq_len_3d)
        self.is_val = bool(is_val)
        self.seed = int(seed)

    def __len__(self):
        return len(self.sequences)

    def _rng_for_index(self, idx):
        s = (self.seed * 1000003) ^ (int(idx) * 9176) ^ 0x9E3779B1
        s = int(s % (2**32 - 1))
        return np.random.RandomState(s)

    def __getitem__(self, idx):
        task_id = None
        if isinstance(idx, (tuple, list)) and len(idx) == 2:
            idx, task_id = idx

        idx = int(idx)
        rec = self.sequences[idx]

        if task_id is None:
            tasks = rec.get("tasks", [])
            if isinstance(tasks, (list, tuple)) and len(tasks) > 0:
                task_id = tasks[0]
            elif tasks is not None:
                task_id = tasks
        if task_id is not None:
            task_id = str(task_id)

        dim = int(rec.get("dim", 2))
        frame_pairs = _extract_frame_pairs(rec)
        if len(frame_pairs) == 0:
            return None

        if dim == 2:
            img_path, msk_path = frame_pairs[0]
            if (not os.path.exists(img_path)) or (not os.path.exists(msk_path)):
                return None

            img_u8 = load_2d_gray_u8(img_path)
            msk_u8 = load_2d_mask01(msk_path)

            img_r = resize_u8_gray(img_u8, self.target_hw)
            msk_r = resize_mask_u8(msk_u8, self.target_hw)

            box = bbox_from_mask(msk_r, pad=self.pad)
            if box is None:
                return None

            img_rgb = np.repeat(img_r[..., None], 3, axis=2)
            img_t = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
            msk_t = torch.from_numpy(msk_r).unsqueeze(0).float()
            box_t = torch.tensor(box, dtype=torch.float32)

            imgs = img_t.unsqueeze(0)
            msks = msk_t.unsqueeze(0)
            boxs = box_t.unsqueeze(0)

            image_ref = img_path
            mask_ref = msk_path
        else:
            valid_frames = []
            for img_path, msk_path in frame_pairs:
                if (not os.path.exists(img_path)) or (not os.path.exists(msk_path)):
                    continue
                try:
                    msk_u8 = load_2d_mask01(msk_path)
                except Exception:
                    continue
                if msk_u8.sum() > 0:
                    valid_frames.append((img_path, msk_path))
        
            if len(valid_frames) == 0:
                return None
        
            # Use ALL foreground frames in order for validation / GIFs
            chosen_pairs = valid_frames
        
            imgs_list, msks_list, box_list = [], [], []
            for img_path, msk_path in chosen_pairs:
                img_u8 = load_2d_gray_u8(img_path)
                msk_u8 = load_2d_mask01(msk_path)
        
                img_r = resize_u8_gray(img_u8, self.target_hw)
                msk_r = resize_mask_u8(msk_u8, self.target_hw)
        
                box = bbox_from_mask(msk_r, pad=self.pad)
                if box is None:
                    continue
        
                img_rgb = np.repeat(img_r[..., None], 3, axis=2)
                img_t = torch.from_numpy(img_rgb).permute(2, 0, 1).float() / 255.0
                msk_t = torch.from_numpy(msk_r).unsqueeze(0).float()
                box_t = torch.tensor(box, dtype=torch.float32)
        
                imgs_list.append(img_t)
                msks_list.append(msk_t)
                box_list.append(box_t)
        
            if len(imgs_list) == 0:
                return None
        
            imgs = torch.stack(imgs_list, 0)
            msks = torch.stack(msks_list, 0)
            boxs = torch.stack(box_list, 0)
        
            image_ref = chosen_pairs[0][0]
            mask_ref = chosen_pairs[0][1]

        meta = {
            "dataset": rec.get("dataset"),
            "subdataset": rec.get("subdataset"),
            "modality": rec.get("modality"),
            "pipeline": rec.get("pipeline", None),
            "identifier": rec.get("identifier", None),
            "class": rec.get("class", None),
            "dim": dim,
            "image": image_ref,
            "mask": mask_ref,
            "pair_index": idx,
            "task_id": task_id,
            "task_label": rec.get("task_label", None),
        }
        return imgs, msks, boxs, meta

def collate_pad_none_seq(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None

    imgs_list, msks_list, boxs_list, metas = zip(*batch)
    T_list = [x.shape[0] for x in imgs_list]
    T_max = max(T_list)

    imgs_pad, msks_pad, boxs_pad, valids = [], [], [], []
    for imgs, msks, boxs, T in zip(imgs_list, msks_list, boxs_list, T_list):
        if T < T_max:
            pad_n = T_max - T
            imgs = torch.cat([imgs, imgs[-1:].repeat(pad_n, 1, 1, 1)], dim=0)
            msks = torch.cat([msks, msks[-1:].repeat(pad_n, 1, 1, 1)], dim=0)
            boxs = torch.cat([boxs, boxs[-1:].repeat(pad_n, 1)], dim=0)

        valid = torch.zeros(T_max, dtype=torch.bool)
        valid[:T] = True

        imgs_pad.append(imgs)
        msks_pad.append(msks)
        boxs_pad.append(boxs)
        valids.append(valid)

    return (
        torch.stack(imgs_pad, 0),
        torch.stack(msks_pad, 0),
        torch.stack(boxs_pad, 0),
        torch.stack(valids, 0),
        list(metas),
    )

# -------------------------------------------------
# Build sampler inputs exactly in the spirit of old Cell 4
# -------------------------------------------------
def sequences_to_sampler_sequences(seqs):
    sampler_seqs = []
    for rec in seqs:
        ds = rec.get("dataset", "unknown")
        sd = rec.get("subdataset", "default")
        task = rec.get("task_id", None) or rec.get("tasks", None) or rec.get("task", None)
        if isinstance(task, (list, tuple)):
            task = task[0] if len(task) > 0 else None
        elif isinstance(task, dict):
            task = task.get("id", None) or task.get("task_id", None) or task.get("name", None)
        if task is None:
            task = f"{ds}/{sd}"
        task = str(task)

        frame_pairs = _extract_frame_pairs(rec)
        if len(frame_pairs) == 0:
            continue

        sampler_seqs.append({
            "dataset": ds,
            "subdataset": sd,
            "tasks": [task],
            "dim": int(rec.get("dim", 2)),
            "modality": rec.get("modality", "unknown"),
            "sequence": frame_pairs,
        })
    return sampler_seqs

def build_tasks_map_from_sequences(seqs):
    tasks_map = {}
    for rec in seqs:
        ds = rec.get("dataset", "unknown")
        sd = rec.get("subdataset", "default")
        task = rec.get("task_id", None) or rec.get("tasks", None) or rec.get("task", None)
        if isinstance(task, (list, tuple)):
            task = task[0] if len(task) > 0 else None
        elif isinstance(task, dict):
            task = task.get("id", None) or task.get("task_id", None) or task.get("name", None)
        if task is None:
            task = f"{ds}/{sd}"
        task = str(task)

        if task not in tasks_map:
            tasks_map[task] = {"classes": [], "datasets": {}}
        tasks_map[task]["datasets"].setdefault(ds, set()).add(sd)

    for t in list(tasks_map.keys()):
        tasks_map[t]["datasets"] = {k: sorted(list(v)) for k, v in tasks_map[t]["datasets"].items()}
    return tasks_map

sampler_val_seqs = sequences_to_sampler_sequences(val_sequences)
tasks_map_val = build_tasks_map_from_sequences(val_sequences)

num_tasks = len(tasks_map_val)
required_num_items = max(int(SUBVAL_NUM_ITEMS), int(num_tasks * SUBVAL_MIN_PER_BUCKET))
actual_num_items = min(required_num_items, len(sampler_val_seqs))

print("\n[SAMPLER INFO]")
print("  total val sequences     :", len(sampler_val_seqs))
print("  discovered task buckets :", num_tasks)
print("  requested num_items     :", SUBVAL_NUM_ITEMS)
print("  adjusted num_items      :", actual_num_items)
print("  min_per_bucket          :", SUBVAL_MIN_PER_BUCKET)

val_ds = OxfordFromTrainingSequences(
    val_sequences,
    target_hw=TARGET_HW,
    pad=5,
    seq_len_3d=SEQ_LEN_3D,
    is_val=True,
    seed=SEED,
)

temp_subval_sampler = BalancedTaskBatchSampler(
    sequences=sampler_val_seqs,
    tasks_map=tasks_map_val,
    batch_size=int(SUBVAL_BATCH_SIZE),
    drop_last=False,
    seed=int(SEED),
    num_samples=int(actual_num_items),
    min_per_bucket=int(SUBVAL_MIN_PER_BUCKET),
)

frozen_subval_batches = list(temp_subval_sampler)
print("  frozen balanced batches :", len(frozen_subval_batches))

class FrozenBatchSampler(torch.utils.data.Sampler):
    def __init__(self, frozen_batches):
        self.frozen_batches = [list(b) for b in frozen_batches]
    def __iter__(self):
        for b in self.frozen_batches:
            yield b
    def __len__(self):
        return len(self.frozen_batches)

subval_loader = DataLoader(
    val_ds,
    batch_sampler=FrozenBatchSampler(frozen_subval_batches),
    num_workers=0,
    pin_memory=False,
    collate_fn=collate_pad_none_seq,
    persistent_workers=False,
)

sb = next(iter(subval_loader))
print("\n[SMOKE BATCH]")
print("  imgs :", sb[0].shape)
print("  msks :", sb[1].shape)
print("  boxes:", sb[2].shape)
print("  valid:", sb[3].shape)
print("  meta :", sb[4][0] if len(sb[4]) else None)

# -------------------------------------------------
# Load Oxford predictor/model
# -------------------------------------------------
assert os.path.exists(SAM_CKPT), f"Missing SAM checkpoint: {SAM_CKPT}"

MEDICAL_REPO = f"{OXFORD_ENV_BASE}/Medical-SAM2"
SITE = f"{OXFORD_ENV_BASE}/site"
for p in [MEDICAL_REPO, SITE]:
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from hydra.core.global_hydra import GlobalHydra
GlobalHydra.instance().clear()
from sam2_train.build_sam import build_sam2_video_predictor

predictor = build_sam2_video_predictor(
    config_file="sam2_hiera_t",
    ckpt_path=SAM_CKPT,
    device="cuda" if torch.cuda.is_available() else "cpu",
    mode="train",
    hydra_overrides_extra=[
        f"model.image_size={TARGET_HW}",
        "model.memory_attention.layer.self_attention.feat_sizes=[16,16]",
        "model.memory_attention.layer.cross_attention.feat_sizes=[16,16]",
    ],
)

if hasattr(predictor, "fill_hole_area"):
    predictor.fill_hole_area = 0
if hasattr(predictor, "model") and hasattr(predictor.model, "fill_hole_area"):
    predictor.model.fill_hole_area = 0

def get_module_for_loading(p):
    if isinstance(p, torch.nn.Module):
        return p
    for attr in ["model", "sam2", "sam", "net", "network"]:
        if hasattr(p, attr) and isinstance(getattr(p, attr), torch.nn.Module):
            return getattr(p, attr)
    for _, v in vars(p).items():
        if isinstance(v, torch.nn.Module):
            return v
    raise RuntimeError("Could not locate underlying torch module inside predictor.")

model = get_module_for_loading(predictor).to(DEVICE).eval()

# -------------------------------------------------
# Resolve checkpoint
# -------------------------------------------------
def _pick_latest_run(parent_dir):
    runs = [d for d in glob.glob(os.path.join(parent_dir, "oxford_refine_*")) if os.path.isdir(d)]
    if len(runs) == 0:
        raise FileNotFoundError(f"No run dirs found under: {parent_dir}")
    runs.sort(key=lambda p: os.path.getmtime(p))
    return runs[-1]

def _pick_ckpt_from_run(run_dir, preference="best"):
    ckpt_dir = os.path.join(run_dir, "checkpoints")
    if preference == "best":
        best_path = os.path.join(ckpt_dir, "best_val.pth")
        if os.path.isfile(best_path):
            return best_path
    epochs = sorted(glob.glob(os.path.join(ckpt_dir, "epoch_*.pth")))
    if len(epochs) == 0:
        raise FileNotFoundError(f"No epoch checkpoints found in: {ckpt_dir}")
    epochs.sort(key=lambda p: os.path.getmtime(p))
    return epochs[-1]

if CHECKPOINT_PATH is None:
    if SPECIFIC_RUN is not None:
        chosen_run = os.path.join(RUN_DIR, SPECIFIC_RUN)
    else:
        chosen_run = _pick_latest_run(RUN_DIR)
    CHECKPOINT_PATH = _pick_ckpt_from_run(chosen_run, preference=CHECKPOINT_PREFERENCE)

assert os.path.isfile(CHECKPOINT_PATH), f"Missing checkpoint: {CHECKPOINT_PATH}"
print("\nUsing checkpoint:", CHECKPOINT_PATH)

ckpt_obj = torch.load(CHECKPOINT_PATH, map_location="cpu")
state_dict = ckpt_obj.get("model", ckpt_obj)
if isinstance(state_dict, dict) and any(k.startswith("module.") for k in state_dict.keys()):
    state_dict = {k.replace("module.", "", 1): v for k, v in state_dict.items()}
missing, unexpected = model.load_state_dict(state_dict, strict=False)
print("Checkpoint loaded. missing:", len(missing), "unexpected:", len(unexpected))

# -------------------------------------------------
# Forward helpers (box-only)
# -------------------------------------------------
def forward_oxford_box(model, imgs_b3hw_01, boxes_b4):
    B, _, H, W = imgs_b3hw_01.shape

    backbone_out = model.forward_image(imgs_b3hw_01)
    _, vision_feats, vision_pos_embeds, feat_sizes = model._prepare_backbone_features(backbone_out)

    feats = [
        feat.permute(1, 2, 0).reshape(B, -1, *sz)
        for feat, sz in zip(vision_feats[::-1], feat_sizes[::-1])
    ][::-1]
    image_embed = feats[-1]
    hires_feats = feats[:-1]

    boxes = boxes_b4.view(B, 2, 2)
    sparse, dense = model.sam_prompt_encoder(points=None, boxes=boxes, masks=None)

    if dense.shape[-2:] != image_embed.shape[-2:]:
        dense = F.interpolate(dense, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)

    image_pe = model.sam_prompt_encoder.get_dense_pe()
    if image_pe.shape[-2:] != image_embed.shape[-2:]:
        image_pe = F.interpolate(image_pe, size=image_embed.shape[-2:], mode="bilinear", align_corners=False)
    image_pe = image_pe.to(image_embed.dtype)

    lowres_logits, _, _, _ = model.sam_mask_decoder(
        image_embeddings=image_embed,
        image_pe=image_pe,
        sparse_prompt_embeddings=sparse,
        dense_prompt_embeddings=dense,
        multimask_output=False,
        repeat_image=False,
        high_res_features=hires_feats,
    )
    logits = F.interpolate(lowres_logits, size=(H, W), mode="bilinear", align_corners=False)
    return logits

def _predictor_target_hw(predictor):
    for attr in ["image_size", "img_size", "target_size"]:
        if hasattr(predictor, attr):
            v = getattr(predictor, attr)
            if isinstance(v, int) and v > 0:
                return int(v), int(v)
    m = getattr(predictor, "model", None)
    if m is not None:
        for attr in ["image_size", "img_size", "target_size"]:
            if hasattr(m, attr):
                v = getattr(m, attr)
                if isinstance(v, int) and v > 0:
                    return int(v), int(v)
        if hasattr(m, "sam_image_embedding_size"):
            emb = int(getattr(m, "sam_image_embedding_size"))
            return emb * 16, emb * 16
    return int(TARGET_HW), int(TARGET_HW)

def _choose_prompt_frames_probabilistic(seq_valid_t, seq_nonempty_t, prompt_prob: float, rng: np.random.RandomState):
    cand = torch.nonzero(seq_valid_t & seq_nonempty_t, as_tuple=False).squeeze(1)
    if cand.numel() == 0:
        return []
    cand_np = cand.detach().cpu().numpy().astype(int)
    keep = rng.rand(cand_np.shape[0]) < float(prompt_prob)
    chosen = cand_np[keep]
    if chosen.size == 0:
        chosen = np.array([cand_np[rng.randint(0, cand_np.shape[0])]], dtype=int)
    chosen = np.unique(chosen)
    chosen.sort()
    return [int(x) for x in chosen.tolist()]

def _seeded_rng_for_val(epoch: int, meta: dict):
    pair_index = meta.get("pair_index", 0)
    try:
        pair_index = int(pair_index)
    except Exception:
        pair_index = 0
    s = (SEED * 1000003) ^ (int(epoch) * 1009) ^ (pair_index * 9176)
    s = int(s % (2**32 - 1))
    return np.random.RandomState(s)

def _build_prompt_frames_by_b_for_val(msks, valid, metas, epoch: int):
    B, T = msks.shape[:2]
    prompt_frames_by_b = []
    for b in range(B):
        meta0 = metas[b] if (isinstance(metas, list) and len(metas) > b) else {}
        rng = _seeded_rng_for_val(epoch, meta0)
        seq_valid = valid[b].detach().cpu().numpy().astype(bool)
        seq_ne = (msks[b].sum(dim=(1, 2, 3)) > 0).detach().cpu().numpy().astype(bool)
        eligible = np.where(seq_valid & seq_ne)[0].astype(int)
        if eligible.size == 0:
            prompt_frames_by_b.append([])
            continue
        if T == 1:
            prompt_frames_by_b.append([int(eligible[0])])
            continue
        chosen = _choose_prompt_frames_probabilistic(valid[b], (msks[b].sum(dim=(1,2,3)) > 0), 0.25, rng)
        prompt_frames_by_b.append(chosen)
    return prompt_frames_by_b

def forward_oxford_video_predictor_sequence_box_only(
    predictor,
    imgs_bt3hw_01,
    boxes_bt4,
    valid_bt=None,
    nonempty_bt=None,
    obj_id=1,
    prompt_frames_by_b=None,
):
    B, T, C, H0, W0 = imgs_bt3hw_01.shape
    Ht, Wt = _predictor_target_hw(predictor)

    imgs = imgs_bt3hw_01
    boxes = boxes_bt4
    if (H0, W0) != (Ht, Wt):
        imgs = imgs.reshape(B * T, 3, H0, W0)
        imgs = F.interpolate(imgs, size=(Ht, Wt), mode="bilinear", align_corners=False)
        imgs = imgs.reshape(B, T, 3, Ht, Wt)

        sx = float(Wt) / float(W0)
        sy = float(Ht) / float(H0)
        boxes = boxes.clone()
        boxes[..., 0] *= sx
        boxes[..., 2] *= sx
        boxes[..., 1] *= sy
        boxes[..., 3] *= sy

    if valid_bt is None:
        valid_bt = torch.ones((B, T), dtype=torch.bool, device=imgs.device)
    else:
        valid_bt = valid_bt.to(device=imgs.device).bool()

    nonempty_bt = nonempty_bt.to(device=imgs.device).bool()
    logits_pred = torch.zeros((B, T, 1, Ht, Wt), dtype=imgs.dtype, device=imgs.device)

    for b in range(B):
        seq_imgs  = imgs[b]
        seq_box   = boxes[b]
        seq_valid = valid_bt[b]
        seq_ne    = nonempty_bt[b]

        if int((seq_valid & seq_ne).sum().item()) == 0:
            continue

        state = predictor.train_init_state(imgs_tensor=seq_imgs)

        prompt_frames = list(prompt_frames_by_b[b]) if prompt_frames_by_b is not None else []
        prompt_frames = [int(f) for f in prompt_frames if 0 <= int(f) < T and bool((seq_valid[int(f)] & seq_ne[int(f)]).item())]
        if len(prompt_frames) == 0:
            cand = torch.nonzero(seq_valid & seq_ne, as_tuple=False).squeeze(1)
            if cand.numel() == 0:
                predictor.reset_state(state)
                continue
            prompt_frames = [int(cand[0].item())]

        for fid in prompt_frames:
            predictor.train_add_new_bbox(
                inference_state=state,
                frame_idx=int(fid),
                obj_id=int(obj_id),
                bbox=seq_box[int(fid)],
                clear_old_points=False,
            )

        video_segments = {}
        for out_fid, out_obj_ids, out_mask_logits in predictor.train_propagate_in_video(state, start_frame_idx=0):
            video_segments[int(out_fid)] = {
                int(oid): out_mask_logits[i] for i, oid in enumerate(out_obj_ids)
            }

        for fid in range(T):
            if not bool(seq_valid[fid].item()):
                continue
            if fid not in video_segments:
                continue
            if int(obj_id) not in video_segments[fid]:
                continue
            x = video_segments[fid][int(obj_id)]
            if x.ndim == 2:
                x = x.unsqueeze(0)
            elif x.ndim == 3 and x.shape[0] != 1:
                x = x[:1]
            logits_pred[b, fid] = x

        predictor.reset_state(state)

    if (Ht, Wt) != (H0, W0):
        logits_out = logits_pred.reshape(B * T, 1, Ht, Wt)
        logits_out = F.interpolate(logits_out, size=(H0, W0), mode="bilinear", align_corners=False)
        logits_out = logits_out.reshape(B, T, 1, H0, W0)
        return logits_out

    return logits_pred

def _forward_batch_box_only(model, predictor, imgs, msks, boxes, valid, metas, epoch):
    B, T = imgs.shape[:2]
    if T == 1:
        img0 = imgs[:, 0]
        box0 = boxes[:, 0]
        logits = forward_oxford_box(model, img0, box0)
        return logits.unsqueeze(1)
    nonempty_bt = (msks.sum(dim=(2, 3, 4)) > 0)
    prompt_frames_by_b = _build_prompt_frames_by_b_for_val(msks=msks, valid=valid, metas=metas, epoch=int(epoch))
    return forward_oxford_video_predictor_sequence_box_only(
        predictor=predictor,
        imgs_bt3hw_01=imgs,
        boxes_bt4=boxes,
        valid_bt=valid,
        nonempty_bt=nonempty_bt,
        obj_id=1,
        prompt_frames_by_b=prompt_frames_by_b,
    )

# -------------------------------------------------
# Validation with full detailed metrics + per-task vis
# -------------------------------------------------
def _safe_tag(s):
    s = "unknown" if s is None else str(s)
    s = re.sub(r"\s+", "_", s.strip())
    s = re.sub(r"[^A-Za-z0-9_\-\.]+", "_", s)
    return s[:120]

def _tb_log_vis(writer, epoch, tag_base, gif_path, img_seq_t3hw, mask_seq_t1hw, logits_seq_thw, fps=6, ping_pong=True):
    frames = None
    try:
        out = save_vis_gif(
            img_seq_t3hw,
            mask_seq_t1hw,
            logits_seq_thw,
            gif_path,
            fps=fps,
            threshold=0.5,
            ping_pong=ping_pong,
        )
        if isinstance(out, (tuple, list)) and len(out) >= 2:
            frames = out[1]
        else:
            frames = out
    except Exception:
        frames = None

    if frames is None:
        try:
            frames = make_vis_frames(
                img_seq_t3hw,
                mask_seq_t1hw,
                logits_seq_thw,
                threshold=0.5,
                ping_pong=ping_pong,
            )
            try:
                save_vis_gif(frames, gif_path, fps=fps)
            except TypeError:
                save_vis_gif(frames, gif_path, fps)
        except Exception as e:
            print("[WARN] Visualization failed:", repr(e))
            return False

    try:
        writer.add_video(
            tag_base,
            frames_to_tb_video(frames),
            global_step=epoch,
            fps=fps,
        )
        return True
    except Exception as e:
        print("[WARN] TensorBoard video logging failed:", repr(e))
        return False

@torch.no_grad()
def validate_epoch_box_only_detailed(model, predictor, loader, epoch=0, writer=None, gif_dir=None):
    model.eval()

    AMP_ENABLED = torch.cuda.is_available()
    AMP_DTYPE = torch.float16 if AMP_ENABLED else None

    task_agg = TaskAggregator(tasks_map={str(k): v for k, v in tasks_map_val.items()})
    mod_agg  = ModalityAggregator(modalities_map=None)

    vol_keys = set(getattr(task_agg, "_vol_keys", set()))

    totals = {}
    counts = {}
    n_slices_total = 0
    n_vols_total   = 0

    visualized_tasks_2d = set()
    visualized_tasks_3d = set()

    pbar = tqdm(loader, desc="Balanced sub-val (box-only)", unit="batch", leave=True)
    
    for bi, batch in enumerate(pbar, 1):
        if batch is None:
            continue

        imgs, msks, boxes, valid, metas = batch
        imgs  = imgs.to(DEVICE, non_blocking=True).float()
        msks  = msks.to(DEVICE, non_blocking=True).float()
        boxes = boxes.to(DEVICE, non_blocking=True).float()
        valid = valid.to(DEVICE, non_blocking=True).bool()

        with torch.cuda.amp.autocast(enabled=AMP_ENABLED, dtype=AMP_DTYPE):
            logits_bt1hw = _forward_batch_box_only(
                model=model,
                predictor=predictor,
                imgs=imgs,
                msks=msks,
                boxes=boxes,
                valid=valid,
                metas=metas,
                epoch=int(epoch),
            )

        probs_bt = torch.sigmoid(logits_bt1hw).float()
        gt_bt    = (msks > 0.5)

        B, T = imgs.shape[:2]
        for b in range(B):
            v_idx = torch.nonzero(valid[b], as_tuple=False).squeeze(1)
            if v_idx.numel() == 0:
                continue

            probs_v = probs_bt[b].index_select(0, v_idx)[:, 0]
            gt_v    = gt_bt[b].index_select(0, v_idx)[:, 0]
            Tv = int(probs_v.shape[0])

            if Tv == 1:
                m = compute_metrics_2d(
                    probs_flat=probs_v,
                    masks_flat=gt_v,
                    idx_list=[0],
                    n_valid=1,
                )
                dim = 2
            else:
                m = compute_metrics_3d(
                    probs=probs_v,
                    gts=gt_v,
                    idx_list=list(range(Tv)),
                    n_valid=Tv,
                    batch={},
                )
                dim = 3

            meta0 = metas[b] if (isinstance(metas, list) and len(metas) > b) else {}
            t_id  = meta0.get("task_id", None)
            t_lab = meta0.get("task_label", None)

            if isinstance(t_id, (list, tuple)):
                t_id = t_id[0] if len(t_id) > 0 else None
            if isinstance(t_lab, (list, tuple)):
                t_lab = t_lab[0] if len(t_lab) > 0 else None
            if t_id is not None:
                t_id = str(t_id)

            task_label = task_agg.label_for(t_id if t_id is not None else t_lab)
            mod_label = str(meta0.get("modality", None) or "unknown")

            task_agg.update(task_label, m, slice_weight=Tv, vol_weight=1, dim=dim, n_items=1)
            mod_agg.update(mod_label,  m, slice_weight=Tv, vol_weight=1, dim=dim, n_items=1)

            for k, v in m.items():
                if not isinstance(v, (int, float, np.floating)):
                    continue
                if v != v:
                    continue
                w = 1.0 if (k in vol_keys) else float(Tv)
                totals[k] = totals.get(k, 0.0) + float(v) * w
                counts[k] = counts.get(k, 0.0) + w

            n_slices_total += Tv
            n_vols_total   += 1
            
            postfix = {
                "slices": n_slices_total,
                "vols": n_vols_total,
            }
            if "dice@0.5" in totals and counts.get("dice@0.5", 0) > 0:
                postfix["dice@0.5"] = f"{(totals['dice@0.5'] / counts['dice@0.5']):.4f}"
            if "iou@0.5" in totals and counts.get("iou@0.5", 0) > 0:
                postfix["iou@0.5"] = f"{(totals['iou@0.5'] / counts['iou@0.5']):.4f}"
            pbar.set_postfix(**postfix)

            safe_task = _safe_tag(task_label)
            
            if dim == 2:
                should_vis = (safe_task not in visualized_tasks_2d)
                vis_suffix = "2d"
            else:
                should_vis = (safe_task not in visualized_tasks_3d)
                vis_suffix = "3d"
            
            if writer is not None and gif_dir is not None and should_vis:
                img_seq = imgs[b].index_select(0, v_idx).detach().cpu()
                mask_seq = msks[b].index_select(0, v_idx).detach().cpu()
                logits_seq = logits_bt1hw[b].index_select(0, v_idx)[:, 0].detach().cpu()
            
                gif_path = os.path.join(gif_dir, f"{safe_task}_{vis_suffix}_epoch{int(epoch):04d}.gif")
                tb_tag = f"subval_box/vis_{vis_suffix}/{safe_task}"
            
                ok = _tb_log_vis(
                    writer=writer,
                    epoch=int(epoch),
                    tag_base=tb_tag,
                    gif_path=gif_path,
                    img_seq_t3hw=img_seq,
                    mask_seq_t1hw=mask_seq,
                    logits_seq_thw=logits_seq,
                    fps=6,
                    ping_pong=True,
                )
                if ok:
                    if dim == 2:
                        visualized_tasks_2d.add(safe_task)
                    else:
                        visualized_tasks_3d.add(safe_task)

    results = {}
    for k in sorted(totals.keys()):
        c = float(counts.get(k, 0.0))
        if c > 0:
            results[k] = float(totals[k] / c)

    results["n_slices"] = int(n_slices_total)
    results["n_volumes"] = int(n_vols_total)

    if writer is not None:
        for k, v in results.items():
            if isinstance(v, (int, float, np.floating)) and (v == v):
                writer.add_scalar(f"subval_box/{k}", float(v), epoch)

        try:
            task_agg.log_to_tensorboard(writer, epoch, split="subval_box")
            writer.add_text("subval_box/per_task_table", "<pre>" + html.escape(task_agg.format_text_table()) + "</pre>", epoch)
        except Exception as e:
            writer.add_text("subval_box/task_agg_error", "<pre>" + html.escape(repr(e)) + "</pre>", epoch)

        try:
            mod_agg.log_to_tensorboard(writer, epoch, split="subval_box")
            writer.add_text("subval_box/per_modality_table", "<pre>" + html.escape(mod_agg.format_text_table()) + "</pre>", epoch)
        except Exception as e:
            writer.add_text("subval_box/mod_agg_error", "<pre>" + html.escape(repr(e)) + "</pre>", epoch)

        writer.add_text(
            "subval_box/vis_summary",
            "<pre>" + html.escape(
                "Visualized 2D tasks (" + str(len(visualized_tasks_2d)) + "):\n"
                + "\n".join(sorted(visualized_tasks_2d))
                + "\n\nVisualized 3D tasks (" + str(len(visualized_tasks_3d)) + "):\n"
                + "\n".join(sorted(visualized_tasks_3d))
            ) + "</pre>",
            epoch,
        )
        writer.flush()

    model.train()
    return results, task_agg, mod_agg, {
        "2d": sorted(visualized_tasks_2d),
        "3d": sorted(visualized_tasks_3d),
    }

# -------------------------------------------------
# Run validation and print full results
# -------------------------------------------------
tb_dir = os.path.join(os.path.dirname(CHECKPOINT_PATH), "..", "tb_eval_subval_standalone_full")
tb_dir = os.path.abspath(tb_dir)
os.makedirs(tb_dir, exist_ok=True)

gif_dir = os.path.join(tb_dir, "gifs_per_task")
os.makedirs(gif_dir, exist_ok=True)

writer = SummaryWriter(log_dir=tb_dir, flush_secs=5, max_queue=10)
epoch_for_eval = int(ckpt_obj.get("epoch", 0)) if isinstance(ckpt_obj, dict) else 0

results, task_agg, mod_agg, visualized_tasks = validate_epoch_box_only_detailed(
    model=model,
    predictor=predictor,
    loader=subval_loader,
    epoch=epoch_for_eval,
    writer=writer,
    gif_dir=gif_dir,
)
writer.close()

print("\n[OVERALL METRICS]")
for k in sorted(results.keys()):
    v = results[k]
    if isinstance(v, (int, float)):
        print(f"  {k:28s} {v:.6f}")
    else:
        print(f"  {k:28s} {v}")

print("\n[PER-TASK TABLE]")
try:
    print(task_agg.format_text_table())
except Exception as e:
    print("Could not format per-task table:", repr(e))

print("\n[PER-MODALITY TABLE]")
try:
    print(mod_agg.format_text_table())
except Exception as e:
    print("Could not format per-modality table:", repr(e))

print("\n[VISUALIZED TASKS]")
for t in visualized_tasks:
    print(" ", t)

print("\nTensorBoard dir:", tb_dir)
print("GIF dir:", gif_dir)